# Implicit-World-Modeling: Mobile UI Transition & Action Predictor

Qwen-family Vision-Language 모델 × 학습 데이터셋 (`AndroidControl_EXP01` / `AndroidControl_EXP02` / `AndroidControl_EXP03` / `AndroidControl_EXP04` / `AndroidControl_EXP05` / `MonkeyCollection`) + `MobiBench` (평가 전용 벤치마크) 매트릭스를 conda env (`implicit-world-modeling`) + LlamaFactory 백엔드로 운영한다.

**Stage 1 / Stage 2 mode (`full` / `lora`)** 은 쉘 스크립트의 `--stage1-mode` · `--stage2-mode` 플래그로 선택한다 (Stage 1 default `full`, Stage 2 default `lora`). 두 모드의 학습 YAML 은 Section 0 에서 자동 생성되며, 산출물 경로 (`adapters/`, `merged/epoch-{E}/`, `eval/.../on-{EVAL_DS}/`) 와 HF Hub repo ID 는 모드별로 분리된다.

**실행 흐름 (Stage 1 / Stage 2 공통): `train → merge → eval`**. eval 은 HF Hub 에 push 된 merged repo 를 pull 해 sweep 하므로 학습 머신이 아닌 환경에서도 재실행 가능하다.


## Dataset Overview

본 프로젝트는 7 개 모바일 UI 인터랙션 데이터셋을 처리한다. 학습 대상 DS 는 **AndroidControl_EXP01 (AC_EXP01)**, **AndroidControl_EXP02 (AC_EXP02)**, **AndroidControl_EXP03 (AC_EXP03)**, **AndroidControl_EXP04 (AC_EXP04)**, **AndroidControl_EXP05 (AC_EXP05)**, **MonkeyCollection (MC)** 여섯 가지이며 (단 MC · AC_EXP04 · AC_EXP05 는 Stage 1 전용 — `_STAGE1_ONLY`), **MobiBench (MB)** 는 OOD 일반화 측정용 평가 전용 벤치마크다.

| DS | 단축 코드 | LlamaFactory prefix | 학습 | Stage 2 | test 형태 | 비고 |
|---|---|---|---|---|---|---|
| AndroidControl_EXP01 | `AC_EXP01` | `IWM-AC_EXP01` | ✓ (S1) | ✓ | id/ood × {state, action} 4 파일 | state_pred + action_pred ratio-mix 학습 (3:7, 5:5, 7:3) |
| AndroidControl_EXP02 | `AC_EXP02` | `IWM-AC_EXP02` | ✓ (S1) | ✓ | id/ood × {state, action} 4 파일 | AC_EXP01 ratio73 + Stage 1 diff loss 실험군 |
| AndroidControl_EXP03 | `AC_EXP03` | `IWM-AC_EXP03` | ✓ (S1) | ✓ | id/ood × {state, action} 4 파일 | AC_EXP01 ratio73 좌표(point) 표현 미러 (index→x,y) |
| AndroidControl_EXP04 | `AC_EXP04` | `IWM-AC_EXP04` | ✓ (S1) | ✗ | id/ood × {state, action} 4 파일 (+ without_open_app 2) | AC_EXP03 멤버십·좌표 표현 + Stage 1 프롬프트 업그레이드 (`_STAGE1_ONLY`) |
| AndroidControl_EXP05 | `AC_EXP05` | `IWM-AC_EXP05` | ✓ (S1) | ✗ | id/ood × {state, action} 4 파일 (+ without_open_app 2) | AC_EXP01 ratio73 을 절대 픽셀 좌표(840×1876)로 미러 + Stage 1 diff loss (`_STAGE1_ONLY`) |
| MonkeyCollection | `MC` | `IWM-MC` | ✓ (S1) | ✗ | 단일 test (random split) | 메타 없음 → Stage 2 자동 skip |
| MobiBench | `MB` | `IWM-MB` | ✗ | ✗ | 단일 파일 (split 없음) | 평가 전용 |

### AndroidControl (원본 source)

대규모 모바일 제어 데이터셋. 본 실험의 주력. Stage 1/2 양쪽 모두에서 **app-level ID/OOD split** 을 단일 partition 으로 공유 — Stage 2 OOD 앱이 Stage 1 train 에 포함되지 않도록 보장한다.

| 항목 | 값 |
|------|-----|
| Stage 1 (World Modeling) | 71,047 건 |
| Stage 2 (Action Prediction) | 91,677 건 |
| Split 도구 | EXP01/EXP02 의 원본 source 자산으로만 사용 (직접 split 하지 않음) |

**Action Type 분포 (Stage 2)** — click 55.7% · finish 16.4% · scroll 11.9% · input 6.6% · open_app 6.1% · navigate_back 3.3% · long_click 0.2%.

### AndroidControl_EXP01 (AC_EXP01)

AndroidControl 의 image 자산 위에서 **state_pred** (다음 화면 XML 예측) 와 **action_pred** (다음 액션 예측) 두 task 를 비율 혼합한 Stage 1 학습 데이터를 제공한다. 비율은 `state:action = 3:7, 5:5, 7:3` 3 종으로 분할되어 각각 다른 학습 가중치를 산출한다 (axis 추가). Test 는 task × split 조합 4 파일.

| 항목 | 값 |
|------|-----|
| Stage 1 train (ratio 3 종) | 각 ~50K (`implicit-world-modeling_stage1_train_{3_7,5_5,7_3}.jsonl`) |
| Stage 1 test | id/ood × {state_pred, action_pred} (4 파일) |
| Filter (선행) | `scripts/filter_long_samples.py --dataset AC_EXP01` — mm-expanded length > `cutoff_len` 제거, Stage 1 두 task + Stage 2 모두 `_filtered.jsonl` 산출 (3 파일). default `image_max_pixels=2097152` 는 Qwen3-VL family 기준 — 다른 family 학습 시 override. |
| Split 도구 | `scripts/split_data.py --dataset AC_EXP01 [--exp01-ratios 3_7,5_5,7_3 --exp01-train-total 50000]` — Stage 1 / Stage 2 모두 `_filtered` 입력만 사용. |
| 채점 | state → `_hungarian_eval.py` (Stage1), action → `_action_eval.py` (Stage2) |
| HF slug | ratio 별 분기: `ac-exp01-ratio37-`, `ac-exp01-ratio55-`, `ac-exp01-ratio73-` |

> 학습 sweep: `--dataset AC_EXP01` 가 ratio 3 종 모두를 자동으로 돌린다 (`--exp01-ratios ratio55,ratio73` 로 부분 실행 가능).
> 평가는 `--train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01` 처럼 단일 ratio 의 가중치를 골라야 한다.

### AndroidControl_EXP03 (AC_EXP03)

AC_EXP01 ratio73 와 **동일한 (episode, step) 멤버십**을 좌표(`point`) 표현으로 미러한 실험군이다. UI 트리 노드가 `index="N"` 대신 `bounds="[x1,y1][x2,y2]" point="[cx,cy]"` 를, 액션이 `index` 대신 `point=[x,y]` (0–1000 정규화) 를 사용한다 — index→좌표 추론 효과만 AC_EXP01 ratio73 대조군과 비교한다.

| 항목 | 값 |
|------|-----|
| Stage 1 train | `implicit-world-modeling_stage1_train.jsonl` (~49,596, AC_EXP01 train_7_3 미러) |
| Stage 1 test | id/ood × {state_pred, action_pred} 4 파일 (+ state_pred_without_open_app 2) |
| Stage 2 | train / test_id / test_ood (~14,881 / 2,971 / 2,952) |
| 원천 | `data/AndroidControl/implicit-world-modeling_stage{1_action,1_state,2}_xy.jsonl` (좌표 표현) |
| 미러 도구 | `python scripts/mirror_experiment.py --experiment exp03` — EXP01 산출 파일별 (episode,step) 키로 좌표 레코드 선택, 이미지 경로는 EXP01 채택. EXP03 원천에 없는 키(~0.8–1.7%)는 제외. |
| HF slug | `ac-exp03-` |

> EXP03 는 EXP01 대비 ~0.8–1.7% 적다 (좌표 변환 시 일부 제외). 각 레코드는 EXP01 레코드와 (episode,step) 1:1 대응하지만 행 수가 달라 eval_viewer cross-compare (byte-identical 전제) 대상은 아니다 — 단독 조회만 지원.

### MonkeyCollection (MC)

Stage 1 (world modeling) 전용 대규모 데이터셋 (약 100K). `episodes_meta.jsonl` 부재 → `split_data.py --dataset MC` 가 random split (기본 0.95) 만 수행. Stage 2 데이터가 존재하지 않으므로 노트북·스크립트 모두 `_STAGE1_ONLY` guard 로 Stage 2 파이프라인에서 자동 skip.

### MobiBench (MB) — 평가 전용 벤치마크

OOD 일반화 측정을 위한 단일 파일 벤치마크. `data/MobiBench/` 아래 `implicit-world-modeling_stage1.jsonl` · `implicit-world-modeling_stage2.jsonl` 두 단일 파일만 존재 (split 없음). `_action_eval.py` / `_hungarian_eval.py` 가 single-pair `overall` 1 섹션으로 채점. `dataset_info.json` 등록은 `scripts/_common.sh::ensure_eval_only_dataset_info()` 가 idempotent 하게 보장 → 노트북 미실행 환경에서도 평가 가능.

교차 평가 실행 (예):
```bash
bash scripts/stage1_eval.sh --model qwen3-vl-8b --train-dataset AC_EXP01 --eval-datasets AC_EXP01,MC,MB
bash scripts/stage2_eval.sh --model qwen3-vl-8b --train-dataset AC_EXP01 --eval-datasets AC_EXP01,MB \
     --stage1-mode full --stage1-epoch 3 --stage2-mode lora
```

> **학습 레시피**: 모든 학습 DS 가 3 epochs (baseline). Stage 1 / Stage 2 의 `full` · `lora` 학습 YAML 은 Section 0 의 YAML 일괄 생성 셀들 (Cell 8 = Stage 1, Cell 10 = Stage 2) 에서 자동 생성된다.
> **AC_EXP01 / AC_EXP02 하이퍼파라미터는 모델 크기 2 단(7-9B / 3-4B) tier 구조로 관리**되며, `_SIZE_CONFIG_AC` (Cell 5) 에 정의된다 — 다만 현재 두 tier 모두 값이 **전부 빈 dict** 이라 실제로는 dataset baseline 이 그대로 적용된다 (Cell 4 참조). MC 는 tier 미적용 — dataset baseline + per-model override 구조.


## 0. Environment Setup

LlamaFactory 워킹트리 부트스트랩은 `scripts/setup_llamafactory.sh` 가 전담한다 — clone → 커밋 pin → `patches/llamafactory/*.patch` 적용 → editable 설치 → 검증.

**선행 조건 (노트북 밖에서 1 회):** conda env 생성/활성화는 여기서 할 수 없다 — `%%bash` 셀은 비대화형 셸이라 `conda activate` 가 **무동작**이고, 그러면 엉뚱한 env 에 조용히 설치된다. 셸에서 먼저 만들고 활성화한 뒤, 그 env 의 커널로 이 노트북을 연다.

```bash
conda create -n implicit-world-modeling python=3.12 -y
conda activate implicit-world-modeling
```

의존성 설치는 아래 셀의 `--install` 이 **활성 env 에** 수행한다 (`pip install -e ".[llamafactory]"` + `pip install -e ./LlamaFactory`). 활성 env 가 `implicit-world-modeling` 이 아니면 스크립트가 설치를 거부한다. `--verify` 는 LF HEAD == pin, 패치 적용 상태, `use_diff_token_weighted_loss` import, `configs/lf_dataset/dataset_info.json` 존재를 표로 확인한다 (실패 시 비0 exit).

> **`BASE_DIR` 을 노트북에서 지정할 필요가 없다** — `implicit_world_modeling.lf_registry` 가 repo 루트를 자동 산출한다 (Cell 5).


In [ ]:
!bash scripts/setup_llamafactory.sh --install --verify


### 크기-tier 하이퍼파라미터 (`_SIZE_CONFIG_AC`, AC_EXP01/AC_EXP02 공유)

AndroidControl 계열 학습 DS (AC_EXP01 / AC_EXP02) 는 모델 크기 tier (`_SIZE_CONFIG_AC` — 현재 **3-4B / 7-9B** 2 단) 의 공유 하이퍼파라미터로 운영하는 구조이며, 그 위에 모델별 delta 만 `hparam_overrides` 로 얹힙니다. MC (MonkeyCollection) 는 tier 미적용 — dataset baseline + per-model override 구조 그대로. MB (MobiBench) 는 평가 전용 벤치마크이므로 학습 하이퍼파라미터 해석에서 제외됩니다.

**크기 배정 (총 4 모델)**:

| 구간 | 모델 |
|---|---|
| **3-4B** | `qwen3-vl-4b`, `qwen2.5-vl-3b` |
| **7-9B** | `qwen3-vl-8b`, `qwen2.5-vl-7b` |

> **아래 lr / LoRA tier 표 3 개는 현재 미적용 — 과거 설계 참고값이다.** `implicit_world_modeling/lf_registry.py` 의 `_SIZE_CONFIG_AC` 는 등록된 모든 tier (`7-9B` / `3-4B`) 에서 `stage1` / `stage1_lora` / `stage2` 가 **모두 빈 dict** 이라, AC 계열 학습 DS 는 **dataset baseline 을 그대로** 적용받는다 (EXP01/EXP02 실측 어댑터와 동일조건을 보존하기 위한 의도된 정책). 따라서 아래 세 표의 값은 **YAML 에 반영되지 않으며** (`2B` 행은 tier 자체가 등록돼 있지도 않다), 활성화하려면 `_SIZE_CONFIG_AC` 를 직접 채워야 한다.

**Stage 1 (full FT)**:

| 구간 | lr | warmup_ratio | max_grad_norm |
|---|---|---|---|
| 2B | 1.5e-5 | 0.08 | 0.5 |
| 3-4B | 1.2e-5 | 0.06 | 0.5 |
| 7-9B | 1.0e-5 (baseline) | 0.03 (baseline) | 1.0 (baseline) |

**Stage 1 LoRA** (`stage1_full` 위에 덮어쓰기):

| 구간 | lr | LoRA r / α | dropout |
|---|---|---|---|
| 2B | 1.5e-4 | 8 / 16 | 0.05 |
| 3-4B | 1.2e-4 | 12 / 24 | 0.05 |
| 7-9B | 1.0e-4 | 16 / 32 | 0.05 |

**Stage 2 (LoRA)**:

| 구간 | lr | LoRA r / α | dropout | warmup | epochs |
|---|---|---|---|---|---|
| 2B | 6.0e-5 | 16 / 32 | 0.05 | 0.05 | 3 |
| 3-4B | 5.0e-5 | 24 / 48 | 0.05 | 0.04 | 3 |
| 7-9B | 4.0e-5 | 32 / 64 | 0.05 | 0.03 | 3 |

> **설계 근거**: `outputs/AC/eval/qwen{2.5-vl-7b,3-vl-8b}/stage2_eval` 실측에서 ① lr 5e-5 가 7-9B 에서 상단 경계 (7B e3 retrograde, 8B cond_text 퇴화), ② dropout 0.10 이 저빈도 action type (input 6.6%, nav_back 3.3%) 을 불안정하게 만듦이 확인됨. 2B / 3-4B 는 Stage 1 크기 규칙을 Stage 2 에 이식한 외삽.

> **참조 우선순위** (merge 순서): `_DATASET_CONFIG` baseline → `_SIZE_CONFIG_AC[size]` (AndroidControl 계열일 때만) → `_MODEL_CONFIG[model].hparam_overrides`. `lf_registry.build_configs()` 가 이 순서로 `dict.update()` 합니다 (Cell 5 가 그 결과인 `CONFIGS` 를 import).


In [ ]:
import os

from dotenv import load_dotenv

# ============================================================
# === 레지스트리 / 정책 SSoT — 이 셀은 thin wrapper 다 ===
# 예전엔 이 셀 본문(28KB)이 _MODEL_CONFIG / _DATASET_CONFIG / _SIZE_CONFIG_AC /
# CONFIGS 빌더 + GPU 정책의 유일본이었다. 지금은 전부 코드로 이관됐고, 이 셀은
# 하류 셀이 쓰는 이름을 노트북 네임스페이스로 재수출하기만 한다.
#
#   레지스트리      implicit_world_modeling/lf_registry.py   (모델 · DS · hparam · CONFIGS)
#   GPU/batch 정책  scripts/gpu_policy.py                    (resolve_gpu_policy)
#   학습 YAML       implicit_world_modeling/gen_configs.py   → configs/train/ (162 YAML, 커밋됨)
#   dataset 등록    configs/lf_dataset/dataset_info.json     (커밋됨)
#
# ★ CONFIGS 엔트리에 per_device_train_batch_size / gradient_accumulation_steps /
#   deepspeed 는 **없다**. 그 셋은 YAML 생성 시점에 resolve_gpu_policy 가 주입한다.
#   따라서 노트북이 들고 있던 GPU 상수(GPU_TYPE / NPROC_PER_NODE / GLOBAL_BATCH_SIZE)와
#   size 별 per-device batch 테이블 · grad_accum 역계산 헬퍼도 전부 은퇴했다 —
#   GPU 조합은 .env 로 scripts/_common.sh 에 전달되고, 학습 시 llamafactory-cli
#   런타임 override (per_device_train_batch_size=N ...) 로 주입된다.
# ============================================================
from implicit_world_modeling.lf_registry import (  # noqa: F401  (하류 셀이 쓰는 이름 재수출)
    BASE_DIR,
    LF_ROOT,
    CONFIGS,
    MODEL_ORDER,
    DS_ORDER,
    MODEL_FAMILY_CONFIG,
    DATASET_MODEL_ELIGIBILITY,
    eligible_models,
    _MODEL_CONFIG,
    _DATASET_CONFIG,
    _SIZE_CONFIG_AC,
    _STAGE1_ONLY,
    _SINGLE_TEST,
    _DUAL_TASK_TEST,
    _EVAL_ONLY_BENCHMARKS,
)

# .env 는 HF_TOKEN 등을 os.environ 에 올린다 (GPU_TYPE / NPROC_PER_NODE 는
# scripts/_common.sh 가 직접 읽으므로 노트북 파이썬에서는 쓰지 않는다).
load_dotenv(os.path.join(BASE_DIR, ".env"))

print(f"Working directory: {os.getcwd()}")
print(f"Project root     : {BASE_DIR}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"Models           : {list(CONFIGS)}")
print(f"Train datasets   : {list(_DATASET_CONFIG)}")
print(f"Stage1-only DS   : {sorted(_STAGE1_ONLY)}")
print(f"Eval-only bench  : {list(_EVAL_ONLY_BENCHMARKS)}")

# --- 실험군별 모델 자격 (lf_registry.DATASET_MODEL_ELIGIBILITY) ---
# 좌표 규약이 family 별로 반전되므로 (Qwen2.5-VL=절대픽셀 / Qwen3-VL=0-1000 정규화)
# 자격 밖 조합은 에러 없이 grounding 만 깨진다 — YAML 생성이 아예 자격만 만든다.
print("\n=== 실험군별 모델 자격 ===")
for _ds in _DATASET_CONFIG:
    print(f"  {_ds:28s} {eligible_models(_ds)}")


### AC_EXP02 diff loss 데이터 준비

`AndroidControl_EXP02` 는 AC_EXP01 ratio73 과 동일 데이터/하이퍼파라미터로 학습하되 **Stage 1 state prediction 에 diff loss (token-weighted SFT)** 를 적용하는 실험군이다.

> **LlamaFactory diff loss 패치는 Cell 3 에서 이미 끝났다** — `scripts/setup_llamafactory.sh` 가 `patches/llamafactory/{0001-diff-loss,0002-double-ce-fix}.patch` 를 적용하고, `--verify` 가 `FinetuningArguments(use_diff_token_weighted_loss=True)` import 로 적용 사실을 증명한다. 예전의 anchor 문자열 치환 스크립트는 은퇴했다 (패치 정본 = `patches/llamafactory/*.patch`).

아래 셀은 **데이터 준비만** 수행한다 (1 회성, 멱등):

1. **AC_EXP02 학습 데이터** — `scripts/diff_loss/preprocess_dataset.py` 가 AC_EXP01 ratio73 train 의 state 토큰에 diff 가중치(ADDED/MODIFIED=2.0, UNCHANGED=1.0)를, action 토큰엔 1.0 을 부여해 `token_weights` 필드를 추가한다.
2. **test / Stage 2 데이터** — AC_EXP01 에서 복사 (AC_EXP01 ratio73 과 동일 평가셋 — 공정 비교).

> 멱등 — 산출물이 이미 있으면 skip. 데이터 전처리는 약 50K 샘플 처리로 수 분~수십 분 소요될 수 있다.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# AC_EXP02 = AC_EXP01 ratio73 + Stage 1 diff loss. 1회성 데이터 준비 (멱등).
# NOTE: LlamaFactory diff loss 패치는 여기서 하지 않는다 — Cell 3 의
#       scripts/setup_llamafactory.sh 가 patches/llamafactory/*.patch 로 적용하고
#       --verify 로 확인한다 (예전의 anchor 문자열 치환 스크립트 호출은 은퇴 —
#       패치 정본은 patches/llamafactory/*.patch 다).
_ac3_dir = Path(BASE_DIR) / "data" / "AndroidControl_EXP01"
_ac4_dir = Path(BASE_DIR) / "data" / "AndroidControl_EXP02"
_ac4_dir.mkdir(parents=True, exist_ok=True)
_dl = Path(BASE_DIR) / "scripts" / "diff_loss"
_env = {**os.environ, "HF_TOKEN": ""}   # 무효 셸 HF_TOKEN 회피 (stored 토큰 사용)

# 1. AC_EXP02 학습 데이터: AC_EXP01 ratio73 train 에 token_weights 부여 (없을 때만)
_ac4_train = _ac4_dir / "implicit-world-modeling_stage1_train.jsonl"
if _ac4_train.exists():
    print(f"[skip] {_ac4_train.name} 이미 존재")
else:
    print("[preprocess] AC_EXP01 ratio73 train -> AC_EXP02 (diff loss token_weights) ...")
    subprocess.run([
        "python", str(_dl / "preprocess_dataset.py"),
        "--input",  str(_ac3_dir / "implicit-world-modeling_stage1_train_7_3.jsonl"),
        "--output", str(_ac4_train),
        "--model",  _MODEL_CONFIG["qwen3-vl-8b"]["model_id"],
        "--w-added", "2.0", "--w-modified", "2.0", "--w-unchanged", "1.0",
    ], check=True, env=_env)

# 2. test / Stage 2 데이터: AC_EXP01 에서 복사 (AC_EXP01 ratio73 과 동일 평가셋)
_shared = [
    "implicit-world-modeling_stage1_test_id_state.jsonl",
    "implicit-world-modeling_stage1_test_ood_state.jsonl",
    "implicit-world-modeling_stage1_test_id_action.jsonl",
    "implicit-world-modeling_stage1_test_ood_action.jsonl",
    "implicit-world-modeling_stage2_train.jsonl",
    "implicit-world-modeling_stage2_test_id.jsonl",
    "implicit-world-modeling_stage2_test_ood.jsonl",
]
for _f in _shared:
    _dst = _ac4_dir / _f
    if _dst.exists():
        print(f"[skip] {_f}")
    else:
        shutil.copy2(_ac3_dir / _f, _dst)
        print(f"[copy] {_f}")
print("[완료] AC_EXP02 데이터 준비 — Section 1/2 의 dataset_info 대조 셀로 진행")


### 학습 YAML — `configs/train/` (repo 소유, 커밋됨)

학습 YAML 은 더 이상 노트북이 `LlamaFactory/examples/custom/` 에 써 넣지 않는다. **정본은 repo 가 소유한다**: `configs/train/IWM-{DS}/stage{1,2}_{full,lora}/*.yaml` (162 개, 커밋됨). `scripts/stage{1,2}_train.sh` 가 이 경로에서 직접 읽는다.

생성기 정본은 `implicit_world_modeling/gen_configs.py` — 레지스트리(`lf_registry`) + GPU 정책(`scripts/gpu_policy.py`) 으로부터 **결정론적으로** 렌더한다.

| 커맨드 | 용도 |
|---|---|
| `python -m implicit_world_modeling.gen_configs --check` | 커밋본과 대조. 불일치/누락/고아 파일이 있으면 diff 출력 후 exit 1 (CI 게이트) |
| `python -m implicit_world_modeling.gen_configs --write` | 재생성. 이후 `git diff configs/train/` 리뷰 필수 |

**생성 대상**: 자격 있는 (모델 × DS × stage × mode) 조합 **전부**. 예전 노트북이 들고 있던 DS allowlist (EXP03/04 YAML 을 덮지 않으려고 AC_EXP05 만 생성하던 보호 필터) 개념은 소멸했다.
- 모델 자격: `lf_registry.DATASET_MODEL_ELIGIBILITY` — EXP03/EXP04 는 Qwen3-VL 계열 전용(0–1000 정규화 좌표), EXP05 는 Qwen2.5-VL 계열 전용(절대 픽셀). 그 외 DS 는 등록 4 모델 전부.
- Stage 2 는 `_STAGE1_ONLY` (MC · AC_EXP04 · AC_EXP05) 를 skip.

**HF repo id 규칙** (`_common.sh`):
- Stage 1 : `SaFD-00/{short}-{slug}world-model-stage1-{full|lora}-epoch{E}`
- Stage 2 base : `SaFD-00/{short}-{slug}base-stage2-{full|lora}-epoch{E2}`
- Stage 2 world-model : `SaFD-00/{short}-{slug}world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}`


#### Stage 1 Training YAML

`configs/train/IWM-{DS}/stage1_{full,lora}/{MODEL}_world-model.yaml` (`IWM-{DS}` = `_DATASET_CONFIG[ds]["lf_subfolder"]`). YAML 내부 경로 (`../outputs/`, `media_dir: ../data`, `deepspeed: examples/...`) 는 **LF_ROOT 기준 상대경로**다 — `llamafactory-cli` 를 `cwd=LlamaFactory` 로 실행하기 때문.

> **DeepSpeed / batch — GPU 분기가 없다**
> 커밋 YAML 은 RTX5090×2 baseline 값 (`per_device_train_batch_size: 1`, `gradient_accumulation_steps: 32`, `deepspeed: examples/deepspeed/ds_z3_offload_config.json`) 으로 고정 emit 된다.
> 예전 셀의 `if GPU_TYPE == "RTX5090": <offload 로 swap>` 조건부 분기는 **금지 패턴**으로 은퇴했다 — RTX5090 이 아닌 경로에서는 한 번도 실행된 적 없는 non-offload 기본값으로 조용히 divergence 했다. 이제 `scripts/gpu_policy.py` 가 GPU 무관 **항상 offload** 를 반환하고, 다른 GPU 조합은 YAML 을 다시 쓰지 않고 `llamafactory-cli train cfg.yaml key=value` 런타임 override 로 주입한다 (`scripts/_common.sh::resolve_overrides`).
> **주의**: offload config 는 DeepSpeed 가 `CPUAdamBuilder` 를 JIT 컴파일하므로 **시스템 CUDA toolkit (nvcc + `cuda.h` 헤더 + `lib64`) 이 torch 의 cu 버전과 정확히 일치**해야 한다 — 불일치 시 `CUDAMismatchException` 으로 학습 시작 전 죽는다 (`scripts/_common.sh` 의 `CUDA_HOME` 정렬 가드 참조).


In [ ]:
# configs/train/** (162 YAML) 이 레지스트리 + GPU 정책으로부터 그대로 재현되는지 대조.
# Stage 1 · Stage 2 를 한 번에 검증한다 (생성기가 두 stage 를 함께 emit).
# 불일치 / 누락 / 고아 파일이 있으면 diff 를 출력하고 exit 1.
!python -m implicit_world_modeling.gen_configs --check


#### Stage 2 Training YAML

`configs/train/IWM-{DS}/stage2_{full,lora}/{MODEL}_{base,world-model-full,world-model-lora}.yaml` (`_STAGE1_ONLY` — MC · AC_EXP04 · AC_EXP05 — 은 skip). world-model variant 는 Stage 1 merged 모델 (HF Hub) 을 `model_name_or_path` 에 적어두지만, `stage2_train.sh` 가 runtime 에 local `merged/{M}_stage1_${STAGE1_MODE}/epoch-${STAGE1_EPOCH}/` 경로로 치환한다.

- Full FT: `finetuning_type: full`, `learning_rate` 는 Full 안정화를 위해 자동으로 낮춤 (기본 1.5e-5, `CONFIGS[...]['stage2']['full_lr']` 로 조정 가능).
- LoRA: `finetuning_type: lora` + `lora_rank/alpha/target/dropout` 블록.

> DeepSpeed / batch 정책은 Stage 1 과 동일 (위 셀 참조 — GPU 분기 없음, 항상 offload).
> **위 Cell 의 `--check` 한 번이 Stage 1 · Stage 2 YAML 을 모두 검증한다** — 생성기가 두 stage 를 함께 emit 하므로 별도의 Stage 2 생성 셀은 없다.


In [ ]:
# 재생성 경로 (레지스트리 / GPU 정책 / 렌더러를 바꿨을 때만):
#
#   1) python -m implicit_world_modeling.gen_configs --write
#   2) git diff configs/train/     ← 의도한 변화인지 리뷰 (as-trained YAML 을 조용히 덮지 않도록)
#   3) python -m pytest tests/test_gen_configs.py
#
# 평시엔 위 셀의 --check 만 돌린다. --write 는 Run All 로 발화하지 않도록 주석 처리해 둔다.
# !python -m implicit_world_modeling.gen_configs --write


## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 Train/Test Split 합니다.

> **LlamaFactory dataset 등록 정본은 `configs/lf_dataset/dataset_info.json`** (커밋됨) 이다 — `scripts/_common.sh` 가 `--dataset_dir <repo>/configs/lf_dataset` 로 LF 에 넘긴다. 아래 **Stage 1 등록 셀은 기록이 아니라 대조**만 한다 (아무것도 쓰지 않는다).
>
> ⚠️ **Section 2 의 Stage 2 등록 셀은 아직 마이그레이션되지 않았다** — 여전히 `LlamaFactory/data/dataset_info.json` 에 in-place 기록하지만, 그 파일은 이제 아무도 읽지 않는다. 그 결과 `IWM-AC_EXP03_stage2_*` 엔트리가 정본에 없어 `stage2_train.sh --dataset AC_EXP03` 는 dataset lookup 에서 죽는다 (데이터와 YAML 은 둘 다 존재한다). 정본에 엔트리를 넣고 커밋해야 한다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)

| File | AndroidControl_EXP01 | MonkeyCollection | MobiBench (eval-only) |
|------|----------------|-------------------|-----------------------|
| implicit-world-modeling_stage1.jsonl | 71,047건 | ~100,000건 | 3,145건 (벤치마크 단일 파일) |
| implicit-world-modeling_stage1_train.jsonl | 50,000 (default) | ~95,000 (95%) | — |
| implicit-world-modeling_stage1_test_id.jsonl | 3,000 (default) | — | — |
| implicit-world-modeling_stage1_test_ood.jsonl | 3,000 (default) | — | — |
| implicit-world-modeling_stage1_test.jsonl | — | ~5,000 (5%) | — |
| Images | (jsonl 내 참조) | (jsonl 내 참조) | (벤치마크 원본) |

**Split 전략:**
- **AC_EXP01** — state_pred / action_pred 두 task 의 ratio mix (3:7, 5:5, 7:3) 학습 + dual-task ID/OOD test.
  원본 source 는 `data/AndroidControl/` (이미지 + jsonl + episodes_meta), 산출물은 `data/AndroidControl_EXP01/`.
  app-level ID/OOD partition 은 Stage 2 와 공유 (`primary_app` 기반).
- **AC_EXP03** — AC_EXP01 ratio73 멤버십을 좌표(point) 표현으로 미러. split 이 아니라 `python scripts/mirror_experiment.py --experiment exp03` 가 EXP01 산출 파일별 (episode,step) 키로 `data/AndroidControl/implicit-world-modeling_stage{1_action,1_state,2}_xy.jsonl` 의 좌표 레코드를 골라 `data/AndroidControl_EXP03/` 에 동일 파일명으로 write (없는 키 제외).
- **MonkeyCollection** — 메타 없음 → random split (seed=42, `--stage1-ratio` 기본 0.95).
- 실행: `python scripts/split_data.py --dataset AC_EXP01` (원본 `data/AndroidControl/` 에서 read; MC 는 `--dataset MC`). AC_EXP03 은 `python scripts/mirror_experiment.py --experiment exp03`.

**MobiBench (eval-only):**
- `data/MobiBench/implicit-world-modeling_stage1.jsonl` 단일 파일이 **평가 세트 전체**로 사용됩니다 (ID/OOD split 없음).
- `stage1_eval.sh --eval-datasets MB` 시 dataset_info 엔트리 `IWM-MB_stage1` 로 로드.


In [ ]:
import json
from pathlib import Path

# dataset_info 정본 = configs/lf_dataset/dataset_info.json (커밋됨).
# 이 셀은 read-only 다 — 레지스트리가 기대하는 엔트리가 정본에 등록돼 있는지,
# 그 entry 의 file_name 이 실제 파일로 resolve 되는지만 대조한다.
# (file_name 은 dataset_dir = configs/lf_dataset/ 기준 상대경로다.)
LF_DATASET_DIR = Path(BASE_DIR) / "configs" / "lf_dataset"
_di_path = LF_DATASET_DIR / "dataset_info.json"
dataset_info = json.loads(_di_path.read_text(encoding="utf-8"))
print(f"[정본] {_di_path}  ({len(dataset_info)} entries)")


def _expected_keys(cfg, ds_name):
    """레지스트리가 이 DS 에 대해 기대하는 dataset_info 키 (Stage 1 + Stage 2)."""
    if ds_name in _DUAL_TASK_TEST or ds_name.startswith("AndroidControl_EXP01"):
        keys = [cfg["ds_s1_train"], cfg["ds_s1_test_id_state"], cfg["ds_s1_test_ood_state"],
                cfg["ds_s1_test_id_action"], cfg["ds_s1_test_ood_action"]]
    elif ds_name in _SINGLE_TEST:
        keys = [cfg["ds_s1_train"], cfg["ds_s1_test"]]
    else:
        keys = [cfg["ds_s1_train"], cfg["ds_s1_test_id"], cfg["ds_s1_test_ood"]]
    if not cfg["stage1_only"]:                      # _STAGE1_ONLY (MC · EXP04 · EXP05) 은 stage2 없음
        keys += [cfg["ds_s2_train"], cfg["ds_s2_test_id"], cfg["ds_s2_test_ood"]]
    return keys


def _audit(key):
    """(상태, 상세) — 등록 여부 + 파일 존재 여부."""
    entry = dataset_info.get(key)
    if entry is None:
        return "미등록", "configs/lf_dataset/dataset_info.json 에 엔트리 없음"
    fpath = (LF_DATASET_DIR / entry["file_name"]).resolve()
    if not fpath.exists():
        return "파일없음", f"{entry['file_name']} — 데이터 미생성 (split_data.py / mirror_experiment.py)"
    return "ok", f"{entry['file_name']}  ({fpath.stat().st_size / 1024 / 1024:.1f} MB)"


_summary = {"ok": 0, "미등록": 0, "파일없음": 0}
_any_model = next(iter(CONFIGS))     # 등록은 모델 무관 — 아무 모델의 DS 뷰나 쓴다.
for ds_name, cfg in CONFIGS[_any_model].items():
    print(f"\n=== {ds_name} ===")
    for key in _expected_keys(cfg, ds_name):
        state, detail = _audit(key)
        _summary[state] += 1
        print(f"  [{state:4s}] {key:38s} {detail}")

# Eval-only 벤치마크 (MobiBench) — 학습엔 안 쓰지만 stage{1,2}_eval.sh 가 이 엔트리로 로드한다.
for bench_name, bcfg in _EVAL_ONLY_BENCHMARKS.items():
    print(f"\n=== {bench_name} (eval-only) ===")
    for key in (bcfg["ds_s1_name"], bcfg["ds_s2_name"]):
        state, detail = _audit(key)
        _summary[state] += 1
        print(f"  [{state:4s}] {key:38s} {detail}")

print(f"\n[요약] ok={_summary['ok']}  미등록={_summary['미등록']}  파일없음={_summary['파일없음']}")
print("  · '미등록' = 정본에 엔트리가 없다 — 데이터 미스테이징 DS (예: AC_EXP04) 이거나 정본이 불완전한 경우다.")
print("  · 데이터 파일은 있는데 '미등록' 이면 후자다 → 그 DS 는 학습이 dataset lookup 에서 죽는다.")
print("  · 엔트리 추가/수정은 configs/lf_dataset/dataset_info.json 을 직접 고쳐 커밋한다 (노트북은 쓰지 않는다).")


In [ ]:
import json
from pathlib import Path

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Dataset Statistics ===\n")

    # MC: _train/_test  AC_EXP01: _train_{ratio} + _test_{id,ood}_{state,action}_pred
    _filenames = [
        "implicit-world-modeling_stage1.jsonl",
        "implicit-world-modeling_stage1_train.jsonl",
        "implicit-world-modeling_stage1_test.jsonl",
        "implicit-world-modeling_stage1_test_id.jsonl",
        "implicit-world-modeling_stage1_test_ood.jsonl",
    ]
    if ds_name.startswith("AndroidControl_EXP01"):
        # ratio variant 3 개가 같은 data_dir 를 공유하므로 중복 출력 방지: ratio37 만 stats 표시.
        if not ds_name.endswith("_ratio37"):
            print("  (other ratio variants share data/AndroidControl_EXP01 — see _ratio37 above)")
            continue
        _filenames = [
            "implicit-world-modeling_stage1_state.jsonl",
            "implicit-world-modeling_stage1_action.jsonl",
            "implicit-world-modeling_stage1_train_3_7.jsonl",
            "implicit-world-modeling_stage1_train_5_5.jsonl",
            "implicit-world-modeling_stage1_train_7_3.jsonl",
            "implicit-world-modeling_stage1_test_id_state.jsonl",
            "implicit-world-modeling_stage1_test_ood_state.jsonl",
            "implicit-world-modeling_stage1_test_id_action.jsonl",
            "implicit-world-modeling_stage1_test_ood_action.jsonl",
        ]
    for fname in _filenames:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            sample = entries[0] if entries else None
            msg_count = len(sample['messages']) if sample else 0
            img_count = len(sample.get('images', [])) if sample else 0
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   Messages per sample: {msg_count}")
            print(f"   Images per sample: {img_count}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
            print()

    img_dir = DATA_PATH / "images"
    if img_dir.exists():
        imgs = list(img_dir.glob("*.png"))
        total_size = sum(f.stat().st_size for f in imgs) / 1024 / 1024 / 1024
        print(f"  Image directory: {img_dir}")
        print(f"   Total images: {len(imgs)}")
        print(f"   Total size: {total_size:.2f} GB")

## 2. Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 **app-level in-domain / out-of-domain split** 으로 분리한 뒤 LlamaFactory 에 등록합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1 과 동일 (공유)

**학습 대상 DS:** AndroidControl_EXP01(AC_EXP01), AndroidControl_EXP02(AC_EXP02), AndroidControl_EXP03(AC_EXP03). MonkeyCollection(MC) 은 Stage 2 데이터/YAML 부재로 제외 — Stage 1 전용.

**AC_EXP01 (ratio sweep):** Stage 1 의 state:action 비율 mix (ratio37/ratio55/ratio73) 산출 모델을 base 로 같은 Stage 2 데이터 (`implicit-world-modeling_stage2_{train,test_id,test_ood}.jsonl`) 로 학습한다. ratio 차원은 stage1 → stage2 계보로만 흐르며 stage2 데이터셋 자체는 ratio 와 무관 — Stage 1 partition 을 그대로 재사용해 OOD app 집합이 두 stage 사이에 일치한다 (`split_data.py::run_ac3_split`).


**Split 워크플로 (AC_EXP01):**
1. 메타데이터 추출 — `scripts/extract_androidcontrol_metadata.py` 로 GCS TFRecord 의 `actions` 중 첫 `open_app` 의 `app_name` 을 추출해 `data/AndroidControl/episodes_meta.jsonl` 생성.
2. Split — `scripts/split_data.py --dataset AC_EXP01` 가 원본 `data/AndroidControl/` 의 jsonl 을 read 해서 앱 집합을 **in-domain / out-of-domain** 으로 나눈 뒤, 각 풀에서 action-type stratified 샘플링 (largest-remainder) → `data/AndroidControl_EXP01/` 에 write.

| 파일 | AndroidControl_EXP01 (기본) |
|------|------------------------------|
| `implicit-world-modeling_stage2_train.jsonl` | 15,000 |
| `implicit-world-modeling_stage2_test_id.jsonl` | 3,000 |
| `implicit-world-modeling_stage2_test_ood.jsonl` | 3,000 |



**MobiBench (eval-only, cross-benchmark):**
- `data/MobiBench/implicit-world-modeling_stage2.jsonl` 단일 파일을 **single-pair overall** 로 채점합니다 (ID/OOD split 없음).
- dataset_info 엔트리: `IWM-MB_stage2`. `stage2_eval.sh --eval-datasets MB` 시 `_action_eval.py score --test ... --pred ...` 경로로 `overall` 섹션 1개만 기록.

**Split 전략 (AC_EXP01):**
- App 단위로 ID / OOD 분리 → 같은 앱 에피소드가 두 test set 에 섞이지 않는다.
- 각 분할 내부는 action type 비율을 원본과 맞추는 stratified subsampling.
- Random seed: 42.


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    # MC 는 Stage 2 데이터가 없으므로 skip.
    if ds_name in _STAGE1_ONLY:
        print(f"[skip] {ds_name}: Stage 1 전용 데이터셋 — Stage 2 entries 등록하지 않음.")
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Data Registration ===\n")

    # AC_EXP01 / AC_EXP02 : train + test_id + test_ood (app-level ID/OOD split)
    if ds_name in _SINGLE_TEST:
        required = [
            ("implicit-world-modeling_stage2_train.jsonl", cfg["ds_s2_train"]),
            ("implicit-world-modeling_stage2_test.jsonl",  cfg["ds_s2_test"]),
        ]
    else:
        required = [
            ("implicit-world-modeling_stage2_train.jsonl",    cfg["ds_s2_train"]),
            ("implicit-world-modeling_stage2_test_id.jsonl",  cfg["ds_s2_test_id"]),
            ("implicit-world-modeling_stage2_test_ood.jsonl", cfg["ds_s2_test_ood"]),
        ]
    for fname, _ds_key in required:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/extract_androidcontrol_metadata.py\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    # AC_EXP01 ratio variant 3 개가 같은 'AndroidControl_EXP01' 데이터를 공유하므로
    # 실제 디렉토리는 단일 (state/action 공유) — file path 는 ds_name 이 아닌 basename(data_dir).
    _data_subdir = "AndroidControl_EXP01" if ds_name.startswith("AndroidControl_EXP01") else ds_name
    DATASETS_TO_REGISTER = {
        ds_key: {
            "file_name": f"../../data/{_data_subdir}/{fname}",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS,
        }
        for fname, ds_key in required
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

# --- Eval-only benchmarks (MobiBench) : 단일 파일 entry (single-pair 평가) ---
# MB Stage 2 는 ID/OOD split 없이 implicit-world-modeling_stage2.jsonl 전체를 overall 1-섹션
# single-pair 모드로 채점한다 (_action_eval.py score --test --pred).
# NOTE: scripts/_common.sh::ensure_eval_only_dataset_info() 가 동일 엔트리를
#       평가 시작 시 자동 보장한다. 이 셀은 노트북 워크플로에서 같은 결과를
#       미리 기록해두는 역할 (idempotent — 두 경로 모두 같은 JSON 을 쓴다).
for bench_name, bcfg in _EVAL_ONLY_BENCHMARKS.items():
    bpath = Path(bcfg["data_dir"]) / bcfg["stage2_jsonl"]
    print(f"\n{'='*60}")
    print(f"=== {bench_name} Stage 2 Benchmark Registration (eval-only) ===\n")
    if not bpath.exists():
        print(f"[skip] {bpath} not found — benchmark file not staged. "
              f"Stage 2 eval on {bench_name} will fail until this file exists.")
        continue
    with open(bpath, 'r') as f:
        count = sum(1 for _ in f)
    print(f"[OK] {bcfg['stage2_jsonl']} ({count} entries, {bpath.stat().st_size / 1024 / 1024:.1f} MB)")
    dataset_info[bcfg["ds_s2_name"]] = {
        "file_name": f"../../data/{bench_name}/{bcfg['stage2_jsonl']}",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS,
    }
    print(f"[Registered] {bcfg['ds_s2_name']} -> ../../data/{bench_name}/{bcfg['stage2_jsonl']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)
print(f"\n[saved] {dataset_info_path}")


In [ ]:
import json
from pathlib import Path
from collections import Counter

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    if ds_name in _STAGE1_ONLY:
        continue
    # AC_EXP01 ratio variant 3 개가 같은 data/AndroidControl_EXP01 를 공유하므로 ratio37 만 1회 출력.
    if ds_name.startswith("AndroidControl_EXP01") and not ds_name.endswith("_ratio37"):
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Dataset Statistics ===\n")

    for fname in ["implicit-world-modeling_stage2.jsonl", "implicit-world-modeling_stage2_train.jsonl", "implicit-world-modeling_stage2_test.jsonl"]:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

            action_types = []
            for entry in entries:
                val = entry['messages'][-1]['value']
                body = val.split('<action>', 1)[-1].split('</action>', 1)[0].strip() if '<action>' in val else val
                try:
                    action = json.loads(body)
                    action_types.append(str(action.get('action_type', action.get('type', 'unknown'))).lower())
                except Exception:
                    action_types.append('parse_error')

            type_counts = Counter(action_types)
            total = len(action_types)
            print(f"   Action type distribution:")
            for atype, count in type_counts.most_common():
                print(f"     {atype}: {count} ({count/total:.1%})")
            print()

In [ ]:
import json
from pathlib import Path
from collections import Counter

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    if ds_name in _STAGE1_ONLY:
        continue
    # AC_EXP01 ratio variant 3 개가 같은 data/AndroidControl_EXP01 를 공유하므로 ratio37 만 1회 출력.
    if ds_name.startswith("AndroidControl_EXP01") and not ds_name.endswith("_ratio37"):
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Action Type Analysis ===")

    train_path = DATA_PATH / "implicit-world-modeling_stage2_train.jsonl"
    test_path = DATA_PATH / "implicit-world-modeling_stage2_test.jsonl"

    for label, fpath in [("Train", train_path), ("Test", test_path)]:
        if not fpath.exists():
            print(f"[SKIP] {label}: {fpath.name} not found")
            continue

        with open(fpath, 'r') as f:
            entries = [json.loads(line) for line in f if line.strip()]

        action_types = []
        for entry in entries:
            val = entry['messages'][-1]['value']
            body = val.split('<action>', 1)[-1].split('</action>', 1)[0].strip() if '<action>' in val else val
            try:
                action = json.loads(body)
                action_types.append(str(action.get('action_type', action.get('type', 'unknown'))).lower())
            except Exception:
                action_types.append('parse_error')

        type_counts = Counter(action_types)
        total = len(action_types)
        print(f"\n  {label} Set Action Types ({total} entries)")
        for atype, count in type_counts.most_common():
            print(f"    {atype}: {count} ({count/total:.1%})")

## 3. Stage 1 SFT Training (Full FT, `qwen3-vl-8b`)

Stage 1 (World Modeling) 학습은 `scripts/stage1_train.sh` 가 담당한다. 본 노트북은 **`qwen3-vl-8b` + `--stage1-mode full`** 단일 변형 walkthrough 만 cell 로 둔다 — 다른 모델 / LoRA 모드는 동일 명령에서 `--model` / `--stage1-mode` 인자만 바꾸면 된다.

**사전 조건**
- `.env` 의 `HF_TOKEN` (Section 4 merge push 용), `NPROC_PER_NODE` (default 2)
- 학습 YAML 은 `configs/train/IWM-{DS}/stage1_full/` 에 커밋돼 있다 (`python -m implicit_world_modeling.gen_configs --check` 로 대조)
- dataset 등록 정본은 `configs/lf_dataset/dataset_info.json` (커밋됨) — `scripts/_common.sh` 가 `--dataset_dir` 로 LF 에 넘긴다

### `scripts/stage1_train.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--dataset` | (필수) | `AC_EXP01` · `AC_EXP02` · `MC` (AC_EXP01 은 `--exp01-ratios` 별 sweep) |
| `--stage1-mode` | `full` | `full` (전 파라미터) 또는 `lora` (PEFT) |
| `--exp01-ratios` | `ratio37,ratio55,ratio73` | AC_EXP01 한정. ratio 별 별도 학습 가중치 산출 |

내부적으로 `FORCE_TORCHRUN=1 NPROC_PER_NODE=$NPROC_PER_NODE llamafactory-cli train ...` 를 model × dataset 곱집합으로 호출한다 (DeepSpeed Z3).

**산출물**: `outputs/{DS}/adapters/{MODEL}_stage1_{MODE}_world-model/checkpoint-{N}/` (epoch 별 checkpoint).

도움말: `bash scripts/stage1_train.sh --help`.

### 3.1 AC_EXP01 (state + action ratio mix)

AC_EXP01 는 `state_pred` / `action_pred` 두 task 를 ratio (state:action) 로 혼합한 train 3 종을 만든다 — ratio 별 학습 가중치가 다르므로 한 번의 호출이 `ratio37 / ratio55 / ratio73` 세 학습을 sweep 한다. `--exp01-ratios ratio55` 식으로 부분 실행도 가능.

In [ ]:
!bash scripts/stage1_train.sh --model qwen3-vl-8b --dataset AC_EXP01 --stage1-mode full --exp01-ratios ratio37,ratio55,ratio73

### 3.2 AC_EXP02 (AC_EXP01 ratio73 + Stage 1 diff loss)

AC_EXP02 는 AC_EXP01 ratio73 과 동일한 학습 데이터·하이퍼파라미터로 학습하되 **Stage 1 state prediction 에 token-weighted diff loss** 를 적용하는 실험군이다 ([README §2.6](README.md#L177)). 데이터는 Section 0 의 "AC_EXP02 diff loss 환경 준비" 셀이 사전 생성해 둔 `data/AndroidControl_EXP02/` 를 사용한다 — 따라서 본 셀 실행 전 Section 0 셀이 한 번은 돌아 있어야 한다 (LF 패치 + train `token_weights` 부여 + AC_EXP01 test/Stage 2 복사). ratio sweep 이 없으므로 단일 학습.

**action_pred 샘플은 `token_weights` 가 전부 1.0 → 기존 cross-entropy 와 동치** 이므로 state_pred 만 diff loss 영향을 받는다.


In [ ]:
!bash scripts/stage1_train.sh --model qwen3-vl-8b --dataset AC_EXP02 --stage1-mode full


### 다른 변형 실행

노트북에는 cell 을 추가하지 말고 인자만 바꿔 실행한다.

- **LoRA mode**: `--stage1-mode lora`. 산출물 경로는 `..._stage1_lora_world-model/`.
- **여러 모델 sweep**: `--model all` (`_common.sh::ALL_MODELS` 4 모델 전부 — `qwen3-vl-8b` · `qwen3-vl-4b` · `qwen2.5-vl-7b` · `qwen2.5-vl-3b`).

## 4. Stage 1 Merge & Upload to HuggingFace (전 epoch push)

`scripts/stage1_merge.sh` 가 Section 3 의 `outputs/{DS}/adapters/.../checkpoint-*/` 를 epoch 별로 export 해 merged weight 를 만들고 HuggingFace Hub 에 push 한다. 본 노트북은 Section 3 와 동일하게 **`qwen3-vl-8b` + `full`** 단일 변형 walkthrough 만 둔다.

### `scripts/stage1_merge.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--model` | `all` | Section 3 와 동일 |
| `--dataset` | `all` | Section 3 와 동일 |
| `--stage1-mode` | `full` | `full` 은 checkpoint 자체를 export, `lora` 는 base + adapter 결합 |
| `--exp01-ratios` | `ratio37,ratio55,ratio73` | AC_EXP01 한정 |

각 `checkpoint-*/` 의 `trainer_state.json.epoch` 을 파싱해 epoch 번호를 추출, 임시 export YAML 을 생성한 뒤 `llamafactory-cli export` 로 merged 디렉토리를 만들고 `huggingface_hub` 로 push (`HF_TOKEN` 필요).

**산출물**
- 로컬: `outputs/{DS}/merged/{MODEL}_stage1_{MODE}_world-model/epoch-{E}/`
- HF Hub: `SaFD-00/{short}-{slug}world-model-stage1-{MODE}-epoch{E}` (예: `SaFD-00/qwen3-vl-8b-ac-world-model-stage1-full-epoch1`)

도움말: `bash scripts/stage1_merge.sh --help`.

### 4.1 AC_EXP01 (ratio mix)

AC_EXP01 는 ratio 별 (`ratio37`, `ratio55`, `ratio73`) 로 별도의 HF repo (`...ac-exp01-ratio37-world-model-stage1-full-epoch{E}` 등) 가 만들어진다.

In [ ]:
!bash scripts/stage1_merge.sh --model qwen3-vl-8b --dataset AC_EXP01 --stage1-mode full --exp01-ratios ratio37,ratio55,ratio73

### 4.2 AC_EXP02 (diff loss 실험군)

AC_EXP02 는 ratio 가 없으므로 단일 merge 호출. HF repo 는 `...ac-exp02-world-model-stage1-full-epoch{E}` slug 로 push 된다.


In [ ]:
!bash scripts/stage1_merge.sh --model qwen3-vl-8b --dataset AC_EXP02 --stage1-mode full


### 다른 변형 실행

- **다른 모델**: `--model {short_name}` (Section 3 와 동일).
- **LoRA mode**: `--stage1-mode lora` — Stage 3 에서 LoRA 로 학습한 adapter 가 있어야 한다.
- **여러 모델 sweep**: `--model all`.

### 5. Stage 1 Evaluation Pipeline

**흐름: `train → merge → eval`.** eval 은 HF Hub 에 push 된 merged repo (`SaFD-00/{short}-{slug}world-model-stage1-{full|lora}-epoch{E}`) 를 pull 해 sweep 한다. 로컬 `adapters/` · `merged/` 디렉토리에 의존하지 않으므로 학습 머신이 아닌 환경에서도 재평가가 성립한다.

1. **Baseline (zero-shot)** — `--variants base` 로 `{model_id}` 자체를 EVAL_DS 마다 추론. 산출: `outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/base/on-{EVAL_DS}/hungarian_metrics.json`.
2. **HF Hub world-model variant sweep** — `--epochs LIST` (기본 `1,2,3`) 로 epoch 별 merged repo 를 pull → `vllm_infer.py` 추론 → `_hungarian_eval.py score` 로 metric 산출.
3. **without_open_app 자동 산출** — 각 `(variant, EVAL_DS)` 마다 정규 score 직후 추론 재실행 없이 `_hungarian_eval.py score --exclude-action open_app --filtered-test-dir ... --filtered-pred-dir on-{EVAL_DS}-without-open_app/` 가 한 번 더 호출되어 GT `open_app` 행을 양쪽에서 동시 drop 한 메트릭과 필터된 jsonl + `predict_results.json` 을 sibling 디렉토리에 idempotent 저장.
4. **EVAL_DS 별 분기** — AC_EXP01 / AC_EXP02 는 dual-task (state/action) + ID/OOD → state→hungarian, action→action metrics. MC / MB 는 단일 파일 → single-pair `overall` 1 섹션. **AC_EXP01 는 dual-task** — state_pred / action_pred 두 task 가 각각 (id, ood) 2 파일을 갖고, `_hungarian_eval.py` (state) + `_action_eval.py` (action) 두 채점기가 별도로 호출되어 `on-AC_EXP01-state/hungarian_metrics.json` + `on-AC_EXP01-action/action_metrics.json` 두 산출물을 만든다. AC_EXP01 학습 모델 평가는 `--exp01-ratio {ratio37|ratio55|ratio73}` 로 ratio 가 정확히 한 개여야 한다.

산출 경로: `outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/{variant}[/epoch-{E}]/on-{EVAL_DS}/hungarian_metrics.json`. 재실행 시 marker 존재 unit 은 정규 / without_open_app 각각 독립 skip.

> **자동 winner 선정 없음** — 사용자가 결과를 보고 Stage 2 에 사용할 epoch 을 `--stage1-epoch` 로 직접 지정 (Section 6 / 7 / 8).


### `scripts/stage1_eval.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--train-dataset` | (필수) | 단일 학습 DS — `AC_EXP01` · `AC_EXP02` · `MC`. AC_EXP01 은 `--exp01-ratio` 필수 |
| `--eval-datasets` | train DS | 콤마 구분 — `AC_EXP01,AC_EXP02,MC,MB` |
| `--model` | `all` | 평가 대상 모델 short name |
| `--stage1-mode` | `full` | HF repo slug 결정 (`...stage1-{MODE}-epoch{E}`) |
| `--variants` | `base,full_world_model,lora_world_model` | `base` (zero-shot), `full_world_model` / `lora_world_model` (HF Hub merged repo pull) |
| `--epochs` | `1,2,3` | merged repo epoch sweep |
| `--exp01-ratio` | `ratio55` | AC_EXP01 단일 ratio (학습 가중치 1 개 ↔ 평가 1 회) |

AC_EXP01 는 dual-task — `state_pred` 는 `_hungarian_eval.py` (Stage 1 채점), `action_pred` 는 `_action_eval.py` (Stage 2 채점) 로 독립 처리하며 산출 디렉토리도 `on-AC_EXP01-state/` · `on-AC_EXP01-action/` 로 분리된다.

**산출물**: `outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/{variant}[/epoch-{E}]/on-{EVAL_DS}/{hungarian_metrics.json | action_metrics.json}` (AC 는 ID/OOD 두 jsonl, 그 외는 단일 파일).

도움말: `bash scripts/stage1_eval.sh --help`. 아래 cell 들은 본 노트북이 정의하는 variant matrix · plotting 코드로, shell 호출은 단일 모델 smoke test 와 cross-benchmark 예시만 둔다.

In [ ]:
import json, math, os
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

# Stage 1 Eval-Loss report — baseline vs per-checkpoint world-model sweep.
# 최신 epoch 기준으로 trainer eval_loss 를 함께 표로 뽑아 참고용 차트 생성.

def _load_eval_loss(p: Path):
    if not p.exists():
        return None
    try:
        with open(p, "r") as f:
            return json.load(f).get("eval_loss")
    except Exception:
        return None

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        ds_code    = cfg["output_prefix"].rstrip("/")
        eval_loss_dir = Path(BASE_DIR) / "outputs" / ds_code / "eval" / (short_name + cfg.get("eval_model_suffix", "")) / "stage1_eval" / "eval_loss"
        base_loss = _load_eval_loss(eval_loss_dir / "base" / "eval_results.json")
        if base_loss is None:
            print(f"[SKIP] {model_key}/{ds_name}: baseline eval_results.json 없음")
            continue

        # 두 모드 각각 per-checkpoint 집계
        mode_rows = {}
        for MODE in ("full", "lora"):
            wm_dir = eval_loss_dir / f"{MODE}_world_model"
            per_ckpt = []
            if wm_dir.is_dir():
                for cp in sorted(wm_dir.glob("epoch-*"),
                                 key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1):
                    l = _load_eval_loss(cp / "eval_results.json")
                    if l is not None:
                        per_ckpt.append((cp.name, l))
            if not per_ckpt:
                continue

            # winner: BEST_CHECKPOINT 참고 (adapter 디렉토리)
            best_file = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}{cfg.get('model_dir_suffix', '')}_stage1_{MODE}" / "BEST_CHECKPOINT"
            winner_name = best_file.read_text().strip() if best_file.exists() else per_ckpt[-1][0]
            mode_rows[MODE] = {"checkpoints": per_ckpt, "winner": winner_name}

        if not mode_rows:
            print(f"[SKIP] {model_key}/{ds_name}: no world-model checkpoint eval_results 발견")
            continue

        base_ppl = math.exp(base_loss)
        print(f"\n=== {model_key} / {ds_name} Stage 1 Eval-Loss ===")
        print(f"Baseline (Zero-shot)  eval_loss={base_loss:.4f}  ppl={base_ppl:.4f}")
        for MODE, info in mode_rows.items():
            print(f"[{MODE}_world_model]")
            for name, l in info["checkpoints"]:
                mark = " <-- winner" if name == info["winner"] else ""
                print(f"  {name:<18} eval_loss={l:.4f}  ppl={math.exp(l):.4f}{mark}")

        # Winner checkpoint 를 대표로 써서 막대그래프 — baseline vs winner (모드별)
        variants = [("Base\n(Zero-shot)", base_loss)]
        for MODE, info in mode_rows.items():
            w_loss = dict(info["checkpoints"])[info["winner"]]
            variants.append((f"{MODE}\n({info['winner']})", w_loss))
        labels, losses = zip(*variants)
        colors = ["#9E9E9E"] + ["#FF5722", "#3F51B5"][:len(variants) - 1]
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        b0 = axes[0].bar(labels, losses, color=colors, width=0.5)
        axes[0].set_title("Eval Loss")
        axes[0].set_ylabel("Cross-Entropy Loss")
        for bar, val in zip(b0, losses):
            axes[0].text(bar.get_x()+bar.get_width()/2, val+max(losses)*0.02, f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        axes[0].set_ylim(0, max(losses) * 1.3); axes[0].grid(axis="y", alpha=0.3)
        perps = [math.exp(l) for l in losses]
        b1 = axes[1].bar(labels, perps, color=colors, width=0.5)
        axes[1].set_title("Perplexity"); axes[1].set_ylabel("Perplexity = exp(loss)")
        for bar, val in zip(b1, perps):
            axes[1].text(bar.get_x()+bar.get_width()/2, val+max(perps)*0.02, f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        axes[1].set_ylim(0, max(perps) * 1.3); axes[1].grid(axis="y", alpha=0.3)
        plt.suptitle(f"Stage 1 Eval-Loss ({model_key} / {ds_name}) — baseline vs winner", fontsize=13, fontweight="bold")
        plt.tight_layout()
        img_path = eval_loss_dir / "stage1_eval_loss_evaluation.png"
        plt.savefig(img_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"[Saved] {img_path}")

        # Markdown report
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        rlines = [f"# Stage 1 Eval-Loss Report: Base vs per-checkpoint sweep ({model_key} / {ds_name})",
                  f"\n> Generated: {now}\n",
                  "## Experiment Setup\n",
                  "| Item | Value |",
                  "|------|-------|",
                  f"| Model | {cfg['model_id']} |",
                  f"| Dataset | {ds_name} |",
                  f"| Test Dataset | {cfg['ds_s1_test']} |",
                  ""]
        rlines.append("## Baseline\n")
        rlines += ["| Metric | Base (Zero-shot) |",
                   "|--------|------------------|",
                   f"| Eval Loss | {base_loss:.4f} |",
                   f"| Perplexity | {base_ppl:.4f} |", ""]
        for MODE, info in mode_rows.items():
            rlines.append(f"## World-Model (stage1_{MODE}) — per-checkpoint\n")
            rlines += [f"| Checkpoint | Eval Loss | Perplexity | vs Base |",
                       f"|------------|-----------|------------|---------|"]
            for name, l in info["checkpoints"]:
                mark = "  **(winner)**" if name == info["winner"] else ""
                rlines.append(f"| {name}{mark} | {l:.4f} | {math.exp(l):.4f} | {(base_loss - l)/base_loss:+.1%} |")
            rlines.append("")
        report = "\n".join(rlines)
        report_path = eval_loss_dir / "evaluation_report.md"
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(report, encoding="utf-8")
        print(f"[Saved] {report_path}")


In [ ]:
## Stage 1 Baseline (zero-shot) — base model 을 각 평가 DS 에 대해 추론.
## 학습 DS 와 무관한 단일 baseline 이므로 --train-dataset 은 AC_EXP01 으로 고정 (산출 경로만 달라짐).
## 평가 DS: AC_EXP01 / AC_EXP02 (dual-task), MC (in-distribution), MB (cross-dataset 벤치마크).
## NOTE: AC_EXP01 는 EVAL_DS 로 들어갈 때 ratio 와 무관 (test 4 파일은 ratio 공유).
##       단, --train-dataset AC_EXP01 자체는 별도 셀 (AC_EXP01 Eval 섹션) 에서 처리.
##
## 산출: outputs/AndroidControl_EXP01/eval/{MODEL}_ratio55/stage1_eval/base/on-{EVAL_DS}/hungarian_metrics.json
!bash scripts/stage1_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01,MC,MB --variants base
## (MC 학습 모델의 baseline 도 별도 기록하려면 주석 해제)
# !bash scripts/stage1_eval.sh --train-dataset MC --eval-datasets AC_EXP01,MC,MB --variants base


In [ ]:
## Stage 1 world-model variant HF Hub sweep.
##
## AC_EXP01 / MC 두 학습 DS 각각에서 HF 에 push 된 merged repo 를
##   SaFD-00/{short}-{slug}world-model-stage1-{full|lora}-epoch{E}
## 를 pull 해 AC_EXP01 / MC (in-distribution) + MB (cross-benchmark) 에서 평가.
## AC_EXP01 는 dual-task (state + action) 두 산출물이 따로 생성된다.
##
## 산출: outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/{full|lora}_world_model/epoch-{E}/on-{EVAL_DS}/hungarian_metrics.json
##
## Winner 자동 선정은 없음. 결과를 보고 Stage 2 의 --stage1-epoch 를 수동 선택.
!bash scripts/stage1_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01,MC,MB \
    --variants full_world_model,lora_world_model --epochs 1,2,3

!bash scripts/stage1_eval.sh --train-dataset MC --eval-datasets AC_EXP01,MC,MB \
    --variants full_world_model,lora_world_model --epochs 1,2,3


In [ ]:
# NOTE: 이 셀은 scripts/_hungarian_eval.py 와 글자 단위 동치를 유지하는 정본 복제다.
# 실제 평가는 `python scripts/_hungarian_eval.py score ...` 또는 stage1_eval.sh 를 통해 실행한다.
# 이 셀은 로직 레퍼런스 / 빠른 디버깅 용도로만 유지된다.

import json
import re
import math
from collections import Counter
from bs4 import BeautifulSoup
from munkres import Munkres

# ── Hungarian Metric 상수 ─────────────────────────────────────────────────
INTERACTIVE_TAGS = {"button", "input", "a", "select", "textarea"}
CONTENT_TAGS     = {"p", "img", "span"}
CLICKABLE_ATTRS  = {"clickable", "long-clickable"}

W_TAG   = 3.0   # tag 불일치 패널티 (매칭 불가)
W_TEXT  = 1.5   # text 불일치
W_INDEX = 0.2   # DOM index 거리

MATCH_THRESHOLD = 1.5   # 이 이상이면 매칭 거부
INDEX_TAU       = 2     # index_acc: 차이 ≤ τ 이면 위치 정확


# ── 요소 추출 (BeautifulSoup) ─────────────────────────────────────────────

def _collect_texts(el):
    """요소 자신 + 자식 전체에서 텍스트 토큰 수집 (중복 제거, 알파벳순 join)."""
    tokens = set()
    def add(v):
        if v:
            tokens.add(v.strip())
    add(el.get("description"))
    add(el.get("id"))
    for child in el.find_all(True):
        add(child.get("description"))
        add(child.get("id"))
        t = child.get_text(strip=True)
        if t:
            tokens.add(t)
    t = el.get_text(strip=True)
    if t:
        tokens.add(t)
    return " | ".join(sorted(tokens)) if tokens else ""


def _safe_int(v, default=-1):
    try:
        return int(v)
    except (TypeError, ValueError):
        return default


def extract_elements(xml_str):
    """XML/HTML 문자열에서 interactive 요소를 평탄화하여 추출."""
    try:
        soup = BeautifulSoup(xml_str, "xml")
    except Exception:
        soup = BeautifulSoup(xml_str, "html.parser")
    elements = []
    for el in soup.find_all(True):
        tag  = el.name
        idx  = _safe_int(el.get("index", -1))
        text = _collect_texts(el)
        is_interactive = tag in INTERACTIVE_TAGS
        is_content     = (tag in CONTENT_TAGS) and bool(text)
        is_clickable   = any(el.get(a) for a in CLICKABLE_ATTRS)
        if is_interactive or is_content or is_clickable:
            elements.append({"tag": tag, "text": text, "index": idx})
    return elements


# ── 비용 함수 ─────────────────────────────────────────────────────────────

def _text_sim(a, b):
    """Jaccard 유사도 (단어 집합 기준)."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    sa = set(a.lower().replace("|", "").split())
    sb = set(b.lower().replace("|", "").split())
    return len(sa & sb) / len(sa | sb)


def _match_cost(e1, e2, max_idx):
    """pred 요소 e1과 gt 요소 e2 사이의 매칭 비용."""
    if e1["tag"] != e2["tag"]:
        return W_TAG
    tc = W_TEXT  * (1.0 - _text_sim(e1["text"], e2["text"]))
    ic = W_INDEX * (abs(e1["index"] - e2["index"]) / max(max_idx, 1))
    return round(tc + ic, 5)


# ── 헝가리안 매칭 ─────────────────────────────────────────────────────────

def _hungarian_match(pred, gt):
    """헝가리안 알고리즘으로 pred-gt 간 최적 1:1 매칭."""
    n, m = len(pred), len(gt)
    if n == 0 or m == 0:
        return [], []
    max_idx = max(
        (e["index"] for e in pred + gt if e["index"] >= 0),
        default=1,
    )
    matrix = [
        [_match_cost(p, g, max_idx) for g in gt]
        for p in pred
    ]
    size   = max(n, m)
    padded = [row + [MATCH_THRESHOLD * 2] * (size - len(row)) for row in matrix]
    while len(padded) < size:
        padded.append([MATCH_THRESHOLD * 2] * size)
    indexes = Munkres().compute(padded)
    pairs = []
    for i, j in indexes:
        if i < n and j < m and matrix[i][j] < MATCH_THRESHOLD:
            pairs.append((i, j, matrix[i][j]))
    return pairs, matrix


def compute_hungarian_acc(pred_str, gt_str):
    """모델 생성 XML과 정답 XML을 비교해 hungarian 기반 평가 메트릭을 반환."""
    _zero = {
        "hungarian_ea": 0.0, "hungarian_f1": 0.0,
        "hungarian_prec": 0.0, "hungarian_rec": 0.0,
        "hungarian_text": 0.0, "hungarian_idx": 0.0,
    }
    try:
        pred_els = extract_elements(pred_str)
        gt_els   = extract_elements(gt_str)
    except Exception:
        return _zero
    if not gt_els:
        return _zero

    pairs, _ = _hungarian_match(pred_els, gt_els)
    n_pred, n_gt, n_matched = len(pred_els), len(gt_els), len(pairs)

    ea   = n_matched / max(n_pred, n_gt) if max(n_pred, n_gt) > 0 else 0.0
    prec = n_matched / n_pred             if n_pred  > 0           else 0.0
    rec  = n_matched / n_gt               if n_gt    > 0           else 0.0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0    else 0.0

    if pairs:
        text_sims = [_text_sim(pred_els[i]["text"], gt_els[j]["text"]) for i, j, _ in pairs]
        idx_diffs = [abs(pred_els[i]["index"] - gt_els[j]["index"]) for i, j, _ in pairs]
        text_avg  = sum(text_sims) / len(text_sims)
        idx_acc   = sum(1 for d in idx_diffs if d <= INDEX_TAU) / len(idx_diffs)
    else:
        text_avg = 0.0
        idx_acc  = 0.0

    return {
        "hungarian_ea":   round(ea, 4),
        "hungarian_f1":   round(f1, 4),
        "hungarian_prec": round(prec, 4),
        "hungarian_rec":  round(rec, 4),
        "hungarian_text": round(text_avg, 4),
        "hungarian_idx":  round(idx_acc, 4),
    }


# ── 텍스트 생성 품질 메트릭 ───────────────────────────────────────────────

def calc_bleu(reference, hypothesis, max_n=4):
    """BLEU-4 score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not hyp_tokens or not ref_tokens:
        return 0.0

    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens)))

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1))
        hyp_ngrams = Counter(tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1))

        clipped = sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in hyp_ngrams.items())
        total = sum(hyp_ngrams.values())

        if total == 0:
            precisions.append(0)
        else:
            precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    return bp * math.exp(log_avg)


def calc_rouge_l(reference, hypothesis):
    """ROUGE-L (F1) score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return 0.0

    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_len = dp[m][n]
    precision = lcs_len / n
    recall = lcs_len / m

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ── Stage 1 전체 평가 함수 ────────────────────────────────────────────────

def evaluate_stage1_predictions(test_path, pred_path):
    """Stage 1 prediction 전체 평가를 수행합니다."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_text = gt_entry['messages'][-1]['value']
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))

        bleu = calc_bleu(gt_text, pred_text)
        rouge_l = calc_rouge_l(gt_text, pred_text)
        exact_match = 1.0 if gt_text.strip() == pred_text.strip() else 0.0
        hungarian = compute_hungarian_acc(pred_text, gt_text)

        results.append({
            'bleu': bleu, 'rouge_l': rouge_l,
            'exact_match': exact_match, 'hungarian': hungarian,
        })

    total = len(results)
    metrics = {
        'total': total,
        'avg_bleu': sum(r['bleu'] for r in results) / total if total else 0,
        'avg_rouge_l': sum(r['rouge_l'] for r in results) / total if total else 0,
        'exact_match_rate': sum(r['exact_match'] for r in results) / total if total else 0,
        'avg_hungarian_ea':   sum(r['hungarian']['hungarian_ea'] for r in results) / total if total else 0,
        'avg_hungarian_f1':   sum(r['hungarian']['hungarian_f1'] for r in results) / total if total else 0,
        'avg_hungarian_prec': sum(r['hungarian']['hungarian_prec'] for r in results) / total if total else 0,
        'avg_hungarian_rec':  sum(r['hungarian']['hungarian_rec'] for r in results) / total if total else 0,
        'avg_hungarian_text': sum(r['hungarian']['hungarian_text'] for r in results) / total if total else 0,
        'avg_hungarian_idx':  sum(r['hungarian']['hungarian_idx'] for r in results) / total if total else 0,
    }
    return metrics, results

print("[Loaded] Stage 1 evaluation functions: calc_bleu, calc_rouge_l, compute_hungarian_acc, evaluate_stage1_predictions")

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path

all_stage1_metrics = {}

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        op = cfg["output_prefix"]
        s1_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / (short_name + cfg.get("eval_model_suffix", "")) / "stage1_eval"
        hung_dir = s1_eval_dir / "hungarian_matching"
        # AC_EXP01 / AC_EXP02 는 dual-task + ID/OOD split 이라 GT 가 여러 파일. 이 셀은 in-notebook 빠른 보기용으로
        # in-domain 파일을 anchor 로 쓴다 (canonical eval 은 stage1_eval.sh + _hungarian_eval.py).
        if ds_name in _STAGE1_ONLY:
            test_path = Path(cfg["data_dir"]) / "implicit-world-modeling_stage1_test.jsonl"
        else:
            test_path = Path(cfg["data_dir"]) / "implicit-world-modeling_stage1_test_id.jsonl"
        train_dir = Path(BASE_DIR) / cfg["save_s1"].lstrip("../")

        MODEL_PREDS = {}
        base_pred = hung_dir / "base" / "generated_predictions.jsonl"
        if base_pred.exists():
            MODEL_PREDS["Baseline (Zero-shot)"] = base_pred
        for p in sorted(hung_dir.glob("epoch-*/generated_predictions.jsonl"),
                        key=lambda x: int(x.parent.name.split("-")[1])):
            MODEL_PREDS[p.parent.name] = p

        combo_key = f"{model_key}/{ds_name}"
        print(f"\n{'='*70}")
        print(f"=== {combo_key} Stage 1 Evaluation ({len(MODEL_PREDS)} model(s)) ===")

        base_loss_path = s1_eval_dir / "eval_loss" / "base" / "eval_results.json"
        fwm_loss_path  = s1_eval_dir / "eval_loss" / "full_world_model" / "eval_results.json"
        if base_loss_path.exists() and fwm_loss_path.exists():
            with open(base_loss_path, 'r') as f: base_loss_metrics = json.load(f)
            with open(fwm_loss_path,  'r') as f: fwm_loss_metrics  = json.load(f)
            print(f"\n  Loss-based Evaluation:")
            print(f"    Base     eval_loss: {base_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(base_loss_metrics['eval_loss']):.4f}")
            print(f"    FWM      eval_loss: {fwm_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(fwm_loss_metrics['eval_loss']):.4f}")

        ds_metrics = {}
        for model_name, pred_path in MODEL_PREDS.items():
            if not pred_path.exists():
                print(f"  [SKIP] {model_name}: {pred_path} missing")
                continue
            metrics, _ = evaluate_stage1_predictions(str(test_path), str(pred_path))
            ds_metrics[model_name] = metrics
            out_metric_path = pred_path.parent / "hungarian_metrics.json"
            with open(out_metric_path, 'w', encoding='utf-8') as f:
                json.dump(metrics, f, ensure_ascii=False, indent=2)

        all_stage1_metrics[combo_key] = ds_metrics

        if ds_metrics:
            print(f"\n  === {combo_key} Stage 1 Comparison ===")
            comparison = pd.DataFrame({
                name: {
                    'BLEU-4': f"{m['avg_bleu']:.4f}", 'ROUGE-L': f"{m['avg_rouge_l']:.4f}",
                    'Exact Match': f"{m['exact_match_rate']:.1%}", 'Hungarian EA': f"{m['avg_hungarian_ea']:.4f}",
                    'Hungarian F1': f"{m['avg_hungarian_f1']:.4f}", 'Hungarian Prec': f"{m['avg_hungarian_prec']:.4f}",
                    'Hungarian Rec': f"{m['avg_hungarian_rec']:.4f}", 'Hungarian Text': f"{m['avg_hungarian_text']:.4f}",
                    'Hungarian Idx': f"{m['avg_hungarian_idx']:.4f}",
                } for name, m in ds_metrics.items()
            })
            display(comparison)

            ckpt_metrics = {k: v for k, v in ds_metrics.items() if k.startswith("checkpoint-")}
            if ckpt_metrics:
                best_name, best_m = max(ckpt_metrics.items(),
                                        key=lambda kv: (kv[1]['avg_hungarian_f1'],
                                                        int(kv[0].split('-')[1])))
                train_dir.mkdir(parents=True, exist_ok=True)
                (train_dir / "BEST_CHECKPOINT").write_text(best_name + "\n", encoding='utf-8')
                summary = {
                    "selected": best_name,
                    "metric": "avg_hungarian_f1",
                    "metric_value": best_m['avg_hungarian_f1'],
                    "candidates": [
                        {"checkpoint": k, "avg_hungarian_f1": v['avg_hungarian_f1'],
                         "avg_bleu": v['avg_bleu'], "avg_rouge_l": v['avg_rouge_l'],
                         "exact_match_rate": v['exact_match_rate']}
                        for k, v in ckpt_metrics.items()
                    ],
                }
                (train_dir / "BEST_CHECKPOINT.json").write_text(
                    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
                print(f"\n  [{combo_key}] Hungarian F1 winner: {best_name} (F1={best_m['avg_hungarian_f1']:.4f})")
                print(f"     -> {train_dir / 'BEST_CHECKPOINT'}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

for combo_key, ds_metrics in all_stage1_metrics.items():
    if not ds_metrics:
        continue
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    model_names = list(ds_metrics.keys())
    n_models = len(model_names)
    colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    text_metrics = ['avg_bleu', 'avg_rouge_l', 'exact_match_rate']
    text_labels = ['BLEU-4', 'ROUGE-L', 'Exact Match']
    x1 = np.arange(len(text_metrics))
    width = 0.8 / n_models

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in text_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[0].bar(x1 + offset, values, width, label=name, color=colors[i])

    axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Score')
    axes[0].set_title('Text Generation Metrics')
    axes[0].set_xticks(x1); axes[0].set_xticklabels(text_labels, rotation=15)
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.0); axes[0].grid(axis='y', alpha=0.3)

    hung_metrics = ['avg_hungarian_ea', 'avg_hungarian_f1', 'avg_hungarian_prec',
                    'avg_hungarian_rec', 'avg_hungarian_text', 'avg_hungarian_idx']
    hung_labels = ['EA', 'F1', 'Precision', 'Recall', 'Text Sim', 'Index Acc']
    x2 = np.arange(len(hung_metrics))

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in hung_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])

    axes[1].set_xlabel('Metric'); axes[1].set_ylabel('Score')
    axes[1].set_title('Hungarian Element Matching')
    axes[1].set_xticks(x2); axes[1].set_xticklabels(hung_labels, rotation=15)
    axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1.0); axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Stage 1 Evaluation ({combo_key})', fontsize=14, fontweight='bold')
    plt.tight_layout()

    op = cfg["output_prefix"]
    s1_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / (short_name + cfg.get("eval_model_suffix", "")) / "stage1_eval"
    save_path = str(s1_eval_dir / "hungarian_matching" / "stage1_hungarian_matching_evaluation.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {save_path}")

In [ ]:
import json, math
from pathlib import Path
from datetime import datetime

# Stage 1 Hungarian Matching report — baseline + per-checkpoint sweep.
# 사전 준비: all_stage1_metrics[f"{model_key}/{ds_name}"] 은 다음 구조여야 한다.
#   {
#     "base":                 <metrics dict>,
#     "full_world_model":     {"<ckpt>": <metrics dict>, ...},   # optional
#     "lora_world_model":     {"<ckpt>": <metrics dict>, ...},   # optional
#   }
# (Cell 57 이 per-checkpoint hungarian_metrics.json 을 이 구조로 적재하도록 갱신됨.)

def _winner_ckpt(adapter_dir: Path, fallback_key: str):
    bc = adapter_dir / "BEST_CHECKPOINT"
    if bc.exists():
        return bc.read_text().strip()
    return fallback_key

def _metric_row(label: str, m: dict, keys):
    cells = [f"{m[key]:{fmt}}" if key in m else "-" for key, _lbl, fmt in keys]
    return f"| {label} | " + " | ".join(cells) + " |"

METRIC_KEYS = [
    ("avg_bleu",          "BLEU-4",         ".4f"),
    ("avg_rouge_l",       "ROUGE-L",        ".4f"),
    ("exact_match_rate",  "Exact Match",    ".1%"),
    ("avg_hungarian_ea",  "Hungarian EA",   ".4f"),
    ("avg_hungarian_f1",  "Hungarian F1",   ".4f"),
    ("avg_hungarian_prec","Hungarian Prec", ".4f"),
    ("avg_hungarian_rec", "Hungarian Rec",  ".4f"),
    ("avg_hungarian_text","Hungarian Text", ".4f"),
    ("avg_hungarian_idx", "Hungarian Idx",  ".4f"),
]

for combo_key, entries in all_stage1_metrics.items():
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    ds_code    = cfg["output_prefix"].rstrip("/")
    s1_eval_dir = Path(BASE_DIR) / "outputs" / ds_code / "eval" / (short_name + cfg.get("eval_model_suffix", "")) / "stage1_eval"
    base_metrics = entries.get("base")

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rlines = [f"# Stage 1 Hungarian Matching Report ({model_key} / {ds_name})",
              f"\n> Generated: {now}\n",
              "## Experiment Setup\n",
              "| Item | Value |",
              "|------|-------|",
              f"| Model | {cfg['model_id']} |",
              f"| Dataset | {ds_name} |",
              f"| Training | {cfg['stage1']['epochs']} epochs, LR={cfg['stage1']['lr']} (cosine) |",
              ""]
    if base_metrics:
        rlines.append("## Baseline (Zero-shot)\n")
        rlines += ["| Metric | Score |", "|--------|-------|"]
        for key, lbl, fmt in METRIC_KEYS:
            if key in base_metrics:
                rlines.append(f"| {lbl} | {base_metrics[key]:{fmt}} |")
        rlines.append("")

    for MODE in ("full", "lora"):
        ckpt_map = entries.get(f"{MODE}_world_model")
        if not ckpt_map:
            continue
        adapter_dir = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}{cfg.get('model_dir_suffix', '')}_stage1_{MODE}"
        winner = _winner_ckpt(adapter_dir, max(ckpt_map.keys(),
                              key=lambda c: int(c.split("-")[-1]) if c.split("-")[-1].isdigit() else -1))
        rlines.append(f"## World-Model (stage1_{MODE}) — per-checkpoint sweep\n")
        header = ["Checkpoint"] + [lbl for _, lbl, _ in METRIC_KEYS]
        rlines.append("| " + " | ".join(header) + " |")
        rlines.append("|" + "|".join(["--------"] * len(header)) + "|")
        for name, m in sorted(ckpt_map.items(),
                              key=lambda kv: int(kv[0].split("-")[-1]) if kv[0].split("-")[-1].isdigit() else -1):
            mark = "  **(winner)**" if name == winner else ""
            row = [name + mark] + [f"{m[key]:{fmt}}" if key in m else "-" for key, _, fmt in METRIC_KEYS]
            rlines.append("| " + " | ".join(row) + " |")
        rlines.append("")

    report = "\n".join(rlines)
    report_path = s1_eval_dir / "hungarian_matching" / "evaluation_report.md"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(report, encoding="utf-8")
    print(f"[Saved] {report_path}")

print("\nDone.")


### Stage 1 AC_EXP01 Eval (state + action 독립 채점)

AC_EXP01 학습 모델의 평가는 ratio 가 정확히 한 개여야 한다 (`--exp01-ratio ratio55` default).
task 별 산출물:
* `outputs/AndroidControl_EXP01/eval/{model}_ratio{NN}/stage1_eval/{variant}/epoch-N/on-AC_EXP01-state/hungarian_metrics.json`
* `outputs/AndroidControl_EXP01/eval/{model}_ratio{NN}/stage1_eval/{variant}/epoch-N/on-AC_EXP01-action/action_metrics.json`


In [ ]:
# AC_EXP01 ratio = r55 학습 결과를 AC_EXP01 test 로 평가 (등록 전 모델 × full+lora world-model + base).
!bash scripts/stage1_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01


In [ ]:
# 단일 모델 / 단일 ratio smoke test.
    --eval-datasets AC_EXP01 --variants base --epochs 1


In [ ]:
# AC_EXP01 학습 모델을 MB 등 다른 벤치마크에도 교차 평가 (ratio 1 개 고정).
!bash scripts/stage1_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio73 \
    --eval-datasets AC_EXP01,MB


### Stage 1 AC_EXP02 Eval (state + action 독립 채점, diff loss 실험군)

AC_EXP02 도 AC_EXP01 와 동일하게 `state_pred` / `action_pred` 두 task 를 각각 채점한다 (`stage1_eval.sh` 가 `AC_EXP02` 입력 시 dual-task helper 로 분기 — [stage1_eval.sh:155-156](scripts/stage1_eval.sh#L155-L156)). 단 ratio sweep 이 없으므로 `--exp01-ratio` 는 사용하지 않는다. 평가 데이터는 AC_EXP01 와 동일 (`data/AndroidControl_EXP02/` 가 AC_EXP01 의 test JSONL 을 그대로 복사) — 대조군 (AC_EXP01 ratio73) 과 같은 test 로 공정 비교한다.


In [ ]:
# AC_EXP02 학습 결과를 AC_EXP02 test 로 평가 (full+lora world-model + base).
!bash scripts/stage1_eval.sh --train-dataset AC_EXP02 --eval-datasets AC_EXP02


In [ ]:
# AC_EXP02 학습 모델을 다른 벤치마크 (MB) 에도 교차 평가.
!bash scripts/stage1_eval.sh --train-dataset AC_EXP02 --eval-datasets AC_EXP02,MB


## 6. Stage 2 SFT Training (LoRA, `qwen3-vl-8b`)

Stage 2 (Action Prediction) 학습은 `scripts/stage2_train.sh` 가 담당한다. Stage 2 는 **`base`** (Stage 1 우회) 와 **`world-model-{stage1_mode}`** (Section 4 의 Stage 1 merged epoch 를 상류로 사용) 두 variant 를 지원하며, 본 노트북은 **`qwen3-vl-8b` + `--stage2-mode lora` + 상류 Stage 1 `full` epoch 1**  단일 변형 walkthrough 만 둔다.

**사전 조건**
- 학습 YAML 은 `configs/train/IWM-{DS}/stage2_lora/` 에 커밋돼 있다 (`python -m implicit_world_modeling.gen_configs --check` 로 대조)
- world-model variant 을 쓰려면 Section 4 가 끝나 있어야 함 (`outputs/{DS}/merged/.../epoch-{E1}/` 존재)
- `_STAGE1_ONLY` (MC · AC_EXP04 · AC_EXP05) 은 Stage 2 미지원. Stage 2 학습 대상 DS 는 `AC_EXP01` (ratio 3 종) · `AC_EXP02` · `AC_EXP03`.

### `scripts/stage2_train.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--model` | `all` | Section 3 와 동일한 short name 레지스트리 |
| `--dataset` | (필수) | `AC_EXP01` · `AC_EXP02` (AC_EXP01 는 ratio mix 3 종 sweep) |
| `--stage2-mode` | `lora` | `lora` (PEFT) 또는 `full` |
| `--stage1-mode` | `full` | 상류 Stage 1 산출 repo 의 mode (`full` / `lora`) |
| `--stage1-epoch` | (world-model variant 시 필수) | 어떤 Stage 1 epoch 의 merged 를 상류로 쓸지 |
| `--exp01-ratios` | `ratio37,ratio55,ratio73` | `--dataset AC_EXP01` 일 때 sweep 할 ratio 부분집합 |

world-model variant 은 runtime 에 YAML 의 `model_name_or_path` 를 `outputs/{OUT_DS}/merged/{MODEL}{SFX}_stage1_{M1}_world-model/epoch-{E1}/` 으로 sed 치환, `__STAGE1_EPOCH__` placeholder 를 epoch 번호로 치환한 뒤 학습. AC_EXP01 ratio variant 는 OUT_DS=AC_EXP01, SFX=_ratio{37,55,73} 로 ratio 별 stage1 merged 를 가리킨다.

**산출물**: `outputs/{OUT_DS}/adapters/{MODEL}{SFX}_stage2_{variant_key}/checkpoint-{N}/` (`variant_key` = `world-model-{stage1_mode}-epoch-{E1}` 또는 `base`). AC_EXP01 는 `outputs/AndroidControl_EXP01/adapters/{MODEL}_ratio{37,55,73}_stage2_*/` 로 ratio 별 분리.

도움말: `bash scripts/stage2_train.sh --help`.

### 6.1 AC_EXP01 (ratio sweep)


In [ ]:
# AC_EXP01 stage2: Stage 1 ratio mix (ratio37/ratio55/ratio73) 의 stage1 merged 를 base 로 같은 stage2 데이터 학습.
# stage2 데이터 자체는 ratio 무관 (implicit-world-modeling_stage2_{train,test_id,test_ood}.jsonl) — Stage 1 partition 재사용.
!bash scripts/stage2_train.sh --model qwen3-vl-8b --dataset AC_EXP01 --stage2-mode lora --stage1-mode full --stage1-epoch 1 --exp01-ratios ratio37,ratio55,ratio73


### 6.2 AC_EXP02 (diff loss 실험군)

AC_EXP02 stage2 데이터는 AC_EXP01 에서 복사한 것이므로 diff loss 가 적용되지 않는다 (Stage 2 는 기존 SFT). ratio sweep 도 없다 — 단일 학습.


In [ ]:
# AC_EXP02 stage2: Stage 1 AC_EXP02 merged (diff loss 학습본) 을 base 로 같은 stage2 데이터 학습.
# stage2 데이터는 AC_EXP01 복사본 — diff loss 미적용 (대조군과 동일 데이터/loss).
!bash scripts/stage2_train.sh --model qwen3-vl-8b --dataset AC_EXP02 --stage2-mode lora --stage1-mode full --stage1-epoch 1


### 다른 변형 실행

- **다른 모델**: `--model {short_name}`.
- **Full FT**: `--stage2-mode full`.
- **다른 Stage 1 계보**: `--stage1-mode lora --stage1-epoch 3` 식으로 상류 epoch / mode 변경.
- **base variant** (Stage 1 우회): `--stage1-epoch` 생략 → YAML 의 base 모델로 직접 학습.

## 7. Stage 2 Merge & Upload to HuggingFace (전 epoch push)

`scripts/stage2_merge.sh` 가 Section 6 의 checkpoint 들을 epoch 별 merged 로 export + HF Hub push 한다. Section 6 와 동일하게 **`qwen3-vl-8b` + `lora` + 상류 Stage 1 `full` epoch 1** 단일 변형 walkthrough.

### `scripts/stage2_merge.sh` 사용법

인자는 `stage2_train.sh` 와 동일 (`--model`, `--dataset`, `--stage2-mode`, `--stage1-mode`, `--stage1-epoch`, `--exp01-ratios`).

**HF repo naming** (`{slug}` 은 AC_EXP01 ratio 별로 다름 — ac-exp01-ratio37-/ac-exp01-ratio55-/ac-exp01-ratio73-)
- base variant: `SaFD-00/{short}-{slug}base-stage2-{M2}-epoch{E2}`
- world-model variant: `SaFD-00/{short}-{slug}world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}` (예: `SaFD-00/qwen3-vl-8b-ac-exp01-ratio55-world-model-stage1-full-epoch1-stage2-lora-epoch3`)

**로컬 산출**: `outputs/{OUT_DS}/merged/{MODEL}{SFX}_stage2_{variant_key}/epoch-{E2}/`. AC_EXP01 는 outputs/AndroidControl_EXP01 단일 부모 + `_ratio{37,55,73}` model suffix 로 ratio 별 분리.

도움말: `bash scripts/stage2_merge.sh --help`.

### 7.1 AC_EXP01 (ratio sweep)


In [ ]:
# AC_EXP01 ratio 별 stage2 adapter 를 merge + HF Hub push. outputs/AndroidControl_EXP01/{adapters,merged}/qwen3-vl-8b_ratio{NN}_stage2_*.
!bash scripts/stage2_merge.sh --model qwen3-vl-8b --dataset AC_EXP01 --stage2-mode lora --stage1-mode full --stage1-epoch 1 --exp01-ratios ratio37,ratio55,ratio73


### 7.2 AC_EXP02 (diff loss 실험군)

AC_EXP02 stage2 는 ratio sweep 없이 단일 학습 산출물만 merge 한다. HF repo 는 `...ac-exp02-...stage2-...` slug 로 push 된다.


In [ ]:
# AC_EXP02 stage2 adapter 를 merge + HF Hub push.
!bash scripts/stage2_merge.sh --model qwen3-vl-8b --dataset AC_EXP02 --stage2-mode lora --stage1-mode full --stage1-epoch 1


### 다른 변형 실행

- **다른 모델 / mode / 상류 계보**: Section 6 와 동일한 인자 교체 규칙.

### 8. Stage 2 Evaluation Pipeline

**흐름: `train → merge → eval`.** eval 은 HF Hub 에 push 된 Stage 2 merged repo 만 pull 하며, 로컬 `adapter/` · `merged/` 디렉토리에 의존하지 않는다. 학습 DS (`--train-dataset`, `AC | AC_EXP01 | AC_EXP02`) 와 평가 DS (`--eval-datasets`, `AC | AC_EXP01 | AC_EXP02 | MB`) 를 분리한다 — MC 는 Stage 2 데이터가 없어 학습 DS 로 거절된다.

**EVAL_DS 별 분기**:
- **AC_EXP01 / AC_EXP02** — ID + OOD 두 test 파일 (`implicit-world-modeling_stage2_test_{id,ood}.jsonl`) 을 함께 추론 → `_action_eval.py score --test-id ... --pred-id ... --test-ood ... --pred-ood ...` 가 `action_metrics.json` 에 `overall` / `in_domain` / `out_of_domain` 3 섹션 기록.
- **MB** — 단일 파일 `implicit-world-modeling_stage2.jsonl` 1 회 추론 → single-pair `overall` 1 섹션.

**Variants**: `base` (zero-shot) · `{full|lora}_base` · `{full|lora}_world_model`. world-model variant 는 `--stage1-mode {full|lora} --stage1-epoch N` 으로 상류 Stage 1 epoch 의 HF 레포 계보 번호를 주입한다.

**HF naming**:
- base: `SaFD-00/{short}-{slug}base-stage2-{M2}-epoch{E2}`
- world-model: `SaFD-00/{short}-{slug}world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}`

**산출 경로**: `outputs/{TRAIN_DS}/eval/{MODEL}/stage2_eval/{variant}[/epoch-{E2}]/on-{EVAL_DS}/action_metrics.json`. 재실행 시 marker 존재 unit 은 variant × EVAL_DS 조합 별로 독립 skip.

> **메트릭 정의** (Step Accuracy): `correct = parse_ok ∧ type==gt.type ∧ field_match(type)`. field_match: `navigate_back` / `finish` 항상 통과, `click` / `long_click` 의 `index`, `scroll.direction`, `open_app.params.app`, `input.params.text` 일치. 정본은 `scripts/_action_eval.py`, 회귀 테스트 `tests/test_action_eval.py` (48 케이스).


### `scripts/stage2_eval.sh` 사용법

| 인자 | 기본값 | 설명 |
| --- | --- | --- |
| `--train-dataset` | (필수) | `AC_EXP01` · `AC_EXP02` (MC 미지원). AC_EXP01 는 `--exp01-ratio` 로 단일 ratio 선택 |
| `--exp01-ratio` | `ratio55` | `--train-dataset AC_EXP01` 일 때 학습 모델의 ratio (`ratio37` / `ratio55` / `ratio73`) |
| `--eval-datasets` | train DS | 콤마 구분 — `AC_EXP01,AC_EXP02,MB` |
| `--model` | `all` | 평가 대상 모델 short name |
| `--variants` | `base,full_base,lora_base,full_world_model,lora_world_model` | base/world-model × full/lora 매트릭스 |
| `--stage1-mode` | `full` | world-model variant 시 상류 Stage 1 mode |
| `--stage1-epoch` | `3` | world-model variant 시 상류 Stage 1 epoch (HF repo slug 의 `epoch{E1}`) |
| `--epochs` | `1,2,3` | Stage 2 merged repo epoch sweep |

채점기는 `_action_eval.py` (Stage 2 Action Prediction). EVAL_DS=`AC_EXP01` · `AC_EXP02` 는 ID + OOD 두 jsonl 을 함께 추론하여 `action_metrics.json` 의 `overall` / `in_domain` / `out_of_domain` 3 섹션 기록, `MB` 는 단일 파일 → `overall` 1 섹션.

**산출물**: `outputs/{OUT_DS}/eval/{MODEL}{SFX}/stage2_eval/{variant}[/epoch-{E2}]/on-{EVAL_DS}/action_metrics.json`. AC_EXP01 / AC_EXP02 는 SFX="" (기존 경로 유지), AC_EXP01 는 OUT_DS=AC_EXP01 + SFX=`_ratio{37,55,73}`.

도움말: `bash scripts/stage2_eval.sh --help`. 아래 cell 들은 본 노트북이 정의하는 baseline + variant sweep · plotting 코드다.

In [ ]:
## Stage 2 Baseline — base model zero-shot 에 대해 in-distribution (AC_EXP01: ID+OOD) + cross-benchmark (MB: overall) 평가.
## EVAL_DS=AC_EXP01 / AC_EXP02 는 _action_eval.py 가 overall/in_domain/out_of_domain 3-section 으로 기록.
## EVAL_DS=MB 는 단일 파일이므로 overall 1-section 만 기록 (single-pair 모드).
##
## 산출: outputs/AndroidControl_EXP01/eval/{MODEL}_ratio55/stage2_eval/base/on-{EVAL_DS}/action_metrics.json
!bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01,MB --variants base


In [ ]:
## Stage 2 base variant HF Hub sweep — SaFD-00/...base-stage2-{M2}-epoch{E2}.
## 매 epoch 별로 in-distribution + cross-benchmark 평가.
##
## 산출: outputs/AndroidControl_EXP01/eval/{MODEL}_ratio55/stage2_eval/{full|lora}_base/epoch-{E2}/on-{EVAL_DS}/action_metrics.json
!bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01,MB \
    --variants full_base,lora_base --epochs 1,2,3


In [ ]:
## Stage 2 world-model variant HF Hub sweep — Stage 1 계보 포함:
##   SaFD-00/...world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}
##
## --stage1-mode / --stage1-epoch 로 상류 Stage 1 merged repo 의 계보 번호를 지정한다.
## 아래 예시는 Stage 1 FULL epoch 3 계보 + Stage 2 full/lora 모두 sweep.
!bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01,MB \
    --stage1-mode full --stage1-epoch 3 \
    --variants full_world_model,lora_world_model --epochs 1,2,3

## Stage 1 LORA 계보도 필요하면 주석 해제:
# !bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01,MB \
#     --stage1-mode lora --stage1-epoch 3 \
#     --variants full_world_model,lora_world_model --epochs 1,2,3


In [ ]:
# NOTE: 이 셀은 scripts/_action_eval.py 와 글자 단위 동치를 유지하는 정본 복제다.
# 실제 평가는 ``python scripts/_action_eval.py score ...`` 또는 stage2_eval.sh 를
# 통해 실행한다. 이 셀은 로직 레퍼런스 / 빠른 디버깅용.

import json
import re
import sys
from collections import defaultdict

# Stage 2 Action Prediction evaluator (정본).
# scripts/_action_eval.py 와 글자 단위 동치를 유지한다.
# 메트릭: Step Accuracy (SA) — bounds 영구 부재 데이터셋용 정의.
#   correct = (parse_ok ∧ type==gt ∧ field_match(type))
# field_match: navigate_back/finish 항상 통과 · click/long_click 의 index ·
#              scroll.direction · open_app.params.app · input.params.text

# ── Action Parsing ───────────────────────────────────────────────────────
def parse_action(text):
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    match = re.search(r'\{[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return None


# ── Field 추출 헬퍼 (top-level + nested params 모두 지원) ────────────────
def _pval(action, key):
    if action is None:
        return None
    if key in action:
        return action[key]
    return (action.get('params') or {}).get(key)


def _norm(s):
    return str(s if s is not None else '').strip().lower()


# ── Step Accuracy 채점 ───────────────────────────────────────────────────
# field_match(type) 가 정의된 type 만 step_correct 로 인정. 그 외 type 은 False.
_FIELD_MATCH_TYPES = {
    'navigate_back', 'navigate_home', 'wait', 'finish',
    'click', 'long_press', 'scroll', 'open_app', 'input_text',
}


def _atype(action):
    """AndroidControl 스키마는 'action_type' 키 사용 (구 'type' fallback)."""
    if action is None:
        return ''
    return str(action.get('action_type', action.get('type', ''))).lower()


def evaluate_single(gt_action, pred_action):
    result = {
        'parsed': pred_action is not None,
        'type_correct': False,
        'step_correct': False,
        'has_index_check': False,    # click / long_click
        'has_dir_check': False,      # scroll
        'has_app_check': False,      # open_app
        'has_text_check': False,     # input
    }
    if pred_action is None:
        return result

    gt_type = _atype(gt_action)
    pred_type = _atype(pred_action)
    result['type_correct'] = (gt_type == pred_type)
    if not result['type_correct']:
        return result

    # field_match
    if gt_type in ('navigate_back', 'navigate_home', 'wait', 'finish'):
        result['step_correct'] = True
        return result

    if gt_type in ('click', 'long_press'):
        result['has_index_check'] = True
        result['step_correct'] = (
            str(gt_action.get('index')) == str(pred_action.get('index'))
        )
        return result

    if gt_type == 'scroll':
        result['has_dir_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'direction')) == _norm(_pval(pred_action, 'direction'))
        )
        return result

    if gt_type == 'open_app':
        result['has_app_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'app_name')) == _norm(_pval(pred_action, 'app_name'))
        )
        return result

    if gt_type == 'input_text':
        result['has_text_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'text')) == _norm(_pval(pred_action, 'text'))
        )
        return result

    # unknown type — type 만 일치해도 step 은 False (정책)
    return result


def evaluate_predictions(test_path, pred_path):
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    if len(gt_entries) != len(pred_entries):
        print(f"[warn] length mismatch: gt={len(gt_entries)} pred={len(pred_entries)} "
              f"→ truncating to {min(len(gt_entries), len(pred_entries))}", file=sys.stderr)

    results = []
    per_type = defaultdict(lambda: {
        'count': 0, 'type_correct': 0, 'step_correct': 0,
    })
    cond = {
        'index': {'n': 0, 'k': 0},   # click + long_click
        'dir':   {'n': 0, 'k': 0},   # scroll
        'app':   {'n': 0, 'k': 0},   # open_app
        'text':  {'n': 0, 'k': 0},   # input
    }

    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_action = parse_action(gt_entry['messages'][-1]['value'])
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))
        pred_action = parse_action(pred_text)

        r = evaluate_single(gt_action, pred_action)
        gt_type = _atype(gt_action) or 'unknown'
        r['gt_type'] = gt_type
        results.append(r)

        per_type[gt_type]['count'] += 1
        per_type[gt_type]['type_correct'] += int(r['type_correct'])
        per_type[gt_type]['step_correct'] += int(r['step_correct'])

        if r['has_index_check']:
            cond['index']['n'] += 1
            cond['index']['k'] += int(r['step_correct'])
        if r['has_dir_check']:
            cond['dir']['n'] += 1
            cond['dir']['k'] += int(r['step_correct'])
        if r['has_app_check']:
            cond['app']['n'] += 1
            cond['app']['k'] += int(r['step_correct'])
        if r['has_text_check']:
            cond['text']['n'] += 1
            cond['text']['k'] += int(r['step_correct'])

    total = len(results)
    parsed = sum(1 for r in results if r['parsed'])
    type_correct = sum(int(r['type_correct']) for r in results)
    step_correct = sum(int(r['step_correct']) for r in results)

    parse_rate = parsed / total if total else 0
    type_acc = type_correct / total if total else 0
    step_acc = step_correct / total if total else 0

    per_type_summary = {}
    for t, d in per_type.items():
        per_type_summary[t] = {
            'count':    d['count'],
            'type_acc': round(d['type_correct'] / d['count'] if d['count'] else 0, 4),
            'step_acc': round(d['step_correct'] / d['count'] if d['count'] else 0, 4),
        }

    macro_step = (sum(v['step_acc'] for v in per_type_summary.values()) /
                  len(per_type_summary)) if per_type_summary else 0

    def _ratio(c):
        return c['k'] / c['n'] if c['n'] else 0

    return {
        'total':                total,
        'parse_rate':           round(parse_rate, 4),
        'type_accuracy':        round(type_acc, 4),
        'step_accuracy':        round(step_acc, 4),
        'macro_step_accuracy':  round(macro_step, 4),
        'cond_index_acc':       round(_ratio(cond['index']), 4),
        'cond_dir_acc':         round(_ratio(cond['dir']),   4),
        'cond_app_acc':         round(_ratio(cond['app']),   4),
        'cond_text_acc':        round(_ratio(cond['text']),  4),
        'per_type':             per_type_summary,
    }

print("[Loaded] Step Accuracy evaluator: parse_action, evaluate_single, evaluate_predictions")


### Stage 2 AC_EXP01 Eval (ratio sweep)

AC_EXP01 ratio 별 학습 모델 (ratio37/ratio55/ratio73) 을 같은 AC_EXP01 stage2 test_id/test_ood 로 sweep. `--exp01-ratio` 로 한 번에 한 ratio 선택. eval 출력은 `outputs/AndroidControl_EXP01/eval/{model}_ratio{NN}/stage2_eval/...`.


In [ ]:
## AC_EXP01 ratio = r55 학습 모델을 AC_EXP01 stage2 test 로 평가.
## stage1 계보(full epoch 3) + stage2 full/lora 전부 sweep.
!bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01 \
    --stage1-mode full --stage1-epoch 3 \
    --variants base,full_base,lora_base,full_world_model,lora_world_model --epochs 1,2,3


In [ ]:
## 다른 ratio (ratio37, ratio73) 도 동일 패턴으로 sweep:
# !bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio37 --eval-datasets AC_EXP01 \
#     --stage1-mode full --stage1-epoch 3 \
#     --variants full_world_model,lora_world_model --epochs 1,2,3
# !bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio73 --eval-datasets AC_EXP01 \
#     --stage1-mode full --stage1-epoch 3 \
#     --variants full_world_model,lora_world_model --epochs 1,2,3

## AC_EXP01 학습 모델 cross-bench 평가 (MB 등):
# !bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio55 --eval-datasets AC_EXP01,MB \
#     --stage1-mode full --stage1-epoch 3 \
#     --variants full_world_model,lora_world_model --epochs 1,2,3


### Stage 2 AC_EXP02 Eval (diff loss 실험군)

AC_EXP02 stage2 데이터는 AC_EXP01 에서 복사한 것 (diff loss 미적용) 이고, ratio sweep 도 없다 — `--exp01-ratio` 인자 자체가 필요 없다. 평가는 AC_EXP01 와 동일하게 ID + OOD 동시 sweep.


In [ ]:
## AC_EXP02 stage2 평가. stage1 계보 (full epoch 3) + stage2 full/lora 전부 sweep.
!bash scripts/stage2_eval.sh --train-dataset AC_EXP02 --eval-datasets AC_EXP02 \
    --stage1-mode full --stage1-epoch 3 \
    --variants base,full_base,lora_base,full_world_model,lora_world_model --epochs 1,2,3


In [ ]:
## AC_EXP01 ratio73 (대조군) 과 직접 비교하려면 같은 인자로 AC_EXP01 도 한 번 더 sweep:
# !bash scripts/stage2_eval.sh --train-dataset AC_EXP01 --exp01-ratio ratio73 --eval-datasets AC_EXP01 \
#     --stage1-mode full --stage1-epoch 3 \
#     --variants full_world_model,lora_world_model --epochs 1,2,3

## AC_EXP02 학습 모델 cross-bench 평가 (MB 등):
# !bash scripts/stage2_eval.sh --train-dataset AC_EXP02 --eval-datasets AC_EXP02,MB \
#     --stage1-mode full --stage1-epoch 3 \
#     --variants full_world_model,lora_world_model --epochs 1,2,3
